In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Conv2D,Flatten,Dense,MaxPooling2D,BatchNormalization,Dropout,MaxPool2D
import matplotlib.pyplot as plt
from keras.applications.vgg19 import VGG19
from keras.applications.vgg16 import VGG16
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import label_binarize

from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras import regularizers
from tensorflow.keras.applications import InceptionV3
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc




In [ ]:

import os
import cv2
import numpy as np

train_folder = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Apple/Train'
test_folder = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Apple/Val'
test_folder1 = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Apple/Test'

IMG_SIZE = (224, 224)
MAX_PER_CLASS = 10000   # 🔴 IMPORTANT

def load_images_limited(folder, max_per_class=None):
    images = []
    labels = []
    
    for label in sorted(os.listdir(folder)):
        label_path = os.path.join(folder, label)
        if not os.path.isdir(label_path):
            continue
        
        count = 0
        for filename in os.listdir(label_path):
            if max_per_class is not None and count >= max_per_class:
                break
            
            img_path = os.path.join(label_path, filename)
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(label)
            count += 1

        print(f"Loaded {count} images from class '{label}'")
    
    return np.array(images), np.array(labels)

# 🔹 TRAIN: only 500 images per class
x_train, y_train = load_images_limited(train_folder, MAX_PER_CLASS)

# 🔹 TEST (no limit)
x_test, y_test = load_images_limited(test_folder)
x_test_1, y_test_1 = load_images_limited(test_folder1)

# 🔹 Verify shapes
print("\nFinal Shapes:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)
print("x_test_1:", x_test_1.shape)
print("y_test_1:", y_test_1.shape)



**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels in y_train
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform the labels in y_test
y_test_encoded = label_encoder.transform(y_test)
y_test_encoded_1 = label_encoder.transform(y_test_1)





In [ ]:
y_test_encoded

# *Creating data set ***

In [ ]:
import random
import shutil
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/Detector_Dataset12345")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}


# ============================================================
# INPUT DATASETS
# ============================================================

DATASETS = {

    # ========================================================
    # CLEAN
    # 80 images per class
    # ========================================================

    "Clean": {
        "path": r"/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Bell Pepper/Train",
        "type": "Clean",
        "images_per_class": 80
    },


    # ========================================================
    # FGSM 0.1
    # 40 images per class
    # ========================================================

    "FGSM_0.1": {
        "path": r"/kaggle/input/datasets/yadavmohit2000/bell-paper-fgsm-pre-post/_08FGSM1_Bell peppert",
        "type": "Adversarial",
        "images_per_class": 40
    },


    # ========================================================
    # FGSM 0.2
    # 40 images per class
    # ========================================================

    "FGSM_0.2": {
        "path": r"/kaggle/input/datasets/yadavmohit2000/bell-paper-fgsm-pre-post/_08FGSM1_Bell peppert",
        "type": "Adversarial",
        "images_per_class": 40
    }

}


# ============================================================
# CREATE OUTPUT FOLDERS
# ============================================================

CLEAN_OUTPUT = OUTPUT_DIR / "Clean"
ADV_OUTPUT = OUTPUT_DIR / "Adversarial"

CLEAN_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

ADV_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FUNCTION: GET IMAGES
# ============================================================

def get_images(folder):

    folder = Path(folder)

    if not folder.exists():
        return []

    return [
        file
        for file in folder.iterdir()
        if file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS
    ]


# ============================================================
# FUNCTION: GET CLASS FOLDERS
# ============================================================

def get_class_folders(main_folder):

    main_folder = Path(main_folder)

    if not main_folder.exists():
        return []

    return sorted(
        [
            folder
            for folder in main_folder.iterdir()
            if folder.is_dir()
        ],
        key=lambda x: x.name
    )


# ============================================================
# GET CLASS NAMES FROM CLEAN DATASET
# ============================================================

clean_path = DATASETS["Clean"]["path"]

clean_class_folders = get_class_folders(
    clean_path
)

if len(clean_class_folders) == 0:

    raise ValueError(
        "No class folders found inside Clean dataset."
    )


class_names = [
    folder.name
    for folder in clean_class_folders
]


print("\n" + "=" * 70)
print("CLASSES FOUND IN CLEAN DATASET")
print("=" * 70)

for class_name in class_names:
    print(class_name)


# ============================================================
# CHECK ALL DATASET PATHS BEFORE PROCESSING
# ============================================================

print("\n" + "=" * 70)
print("DATASET PATH CHECK")
print("=" * 70)

for dataset_name, dataset_info in DATASETS.items():

    path = Path(dataset_info["path"])

    print(f"\n{dataset_name}")
    print(f"Path   : {path}")
    print(f"Exists : {path.exists()}")

    if path.exists():

        folders = get_class_folders(path)

        print(
            f"Classes found: {len(folders)}"
        )

        for folder in folders:

            images = get_images(folder)

            print(
                f"   {folder.name} "
                f"-> {len(images)} images"
            )


# ============================================================
# PROCESS DATASETS
# ============================================================

for dataset_name, dataset_info in DATASETS.items():

    main_path = Path(
        dataset_info["path"]
    )

    dataset_type = dataset_info["type"]

    images_per_class = dataset_info[
        "images_per_class"
    ]


    print("\n" + "=" * 70)
    print(f"PROCESSING: {dataset_name}")
    print(
        f"Images per class: {images_per_class}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Check main folder
    # --------------------------------------------------------

    if not main_path.exists():

        print(
            f"ERROR: Dataset path does not exist:"
        )

        print(main_path)

        continue


    # --------------------------------------------------------
    # Process each class
    # --------------------------------------------------------

    for class_name in class_names:

        class_folder = (
            main_path / class_name
        )


        # ----------------------------------------------------
        # Check class folder
        # ----------------------------------------------------

        if not class_folder.exists():

            print(
                f"WARNING: {dataset_name} "
                f"does not contain:"
            )

            print(
                f"   {class_name}"
            )

            continue


        # ----------------------------------------------------
        # Get images
        # ----------------------------------------------------

        images = get_images(
            class_folder
        )


        print(
            f"{class_name}: "
            f"{len(images)} available "
            f"| Required: {images_per_class}"
        )


        # ----------------------------------------------------
        # Check minimum number
        # ----------------------------------------------------

        if len(images) < images_per_class:

            print(
                f"WARNING: Not enough images "
                f"in {dataset_name}/{class_name}"
            )

            continue


        # ----------------------------------------------------
        # Random selection
        # ----------------------------------------------------

        selected_images = random.sample(
            images,
            images_per_class
        )


        # ----------------------------------------------------
        # Copy images
        # ----------------------------------------------------

        for i, image_path in enumerate(
            selected_images,
            start=1
        ):


            # =================================================
            # CLEAN
            # =================================================

            if dataset_type == "Clean":

                new_name = (
                    f"{class_name}_"
                    f"{i:05d}"
                    f"{image_path.suffix.lower()}"
                )

                destination = (
                    CLEAN_OUTPUT /
                    new_name
                )


            # =================================================
            # ADVERSARIAL
            # =================================================

            else:

                new_name = (
                    f"{dataset_name}_"
                    f"{class_name}_"
                    f"{i:05d}"
                    f"{image_path.suffix.lower()}"
                )

                destination = (
                    ADV_OUTPUT /
                    new_name
                )


            shutil.copy2(
                image_path,
                destination
            )


# ============================================================
# COUNT CLEAN
# ============================================================

clean_count = len([
    f
    for f in CLEAN_OUTPUT.iterdir()
    if f.is_file()
    and f.suffix.lower() in IMAGE_EXTENSIONS
])


# ============================================================
# COUNT ADVERSARIAL
# ============================================================

adv_count = len([
    f
    for f in ADV_OUTPUT.iterdir()
    if f.is_file()
    and f.suffix.lower() in IMAGE_EXTENSIONS
])


# ============================================================
# ATTACK-WISE COUNTS
# ============================================================

fgsm_01_count = len([
    f
    for f in ADV_OUTPUT.iterdir()
    if f.is_file()
    and f.name.startswith("FGSM_0.1_")
])


fgsm_02_count = len([
    f
    for f in ADV_OUTPUT.iterdir()
    if f.is_file()
    and f.name.startswith("FGSM_0.2_")
])


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("DATASET PREPARATION COMPLETED")
print("=" * 70)

print(
    f"Clean images       : {clean_count}"
)

print(
    f"FGSM 0.1 images    : {fgsm_01_count}"
)

print(
    f"FGSM 0.2 images    : {fgsm_02_count}"
)

print(
    f"Adversarial total  : {adv_count}"
)

print(
    f"Total images       : "
    f"{clean_count + adv_count}"
)


# ============================================================
# EXPECTED RESULT
# ============================================================

print("\n")
print("=" * 70)
print("EXPECTED RESULT FOR 6 CLASSES")
print("=" * 70)

print("Clean       : 6 × 80 = 480")
print("FGSM 0.1    : 6 × 40 = 240")
print("FGSM 0.2    : 6 × 40 = 240")
print("--------------------------------")
print("Adversarial : 480")
print("Total       : 960")


# ============================================================
# OUTPUT LOCATIONS
# ============================================================

print("\n")
print("=" * 70)
print("OUTPUT LOCATIONS")
print("=" * 70)

print(
    "Clean folder:"
)

print(
    CLEAN_OUTPUT
)

print(
    "\nAdversarial folder:"
)

print(
    ADV_OUTPUT
)

In [ ]:
import shutil

source = "/kaggle/working/Detector_Dataset12345"

shutil.make_archive(
    "/kaggle/working/Adversarial",
    "zip",
    root_dir=source
)

print("ZIP created:")
print("/kaggle/working/Adversarial.zip")

# Image_level_KNN training for each ADV_data of each crops one be one ***

In [ ]:
import os
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler


# ============================================================
# SETTINGS
# ============================================================

DATASET_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/combine-tomato"
)

IMAGE_SIZE = (224, 224)

RANDOM_STATE = 42

# KNN
K = 5


# ============================================================
# FOLDERS
# ============================================================

CLEAN_FOLDER = DATASET_PATH / "Clean"
ADV_FOLDER = DATASET_PATH / "Adversarial"


print("Clean folder:")
print(CLEAN_FOLDER)

print("\nAdversarial folder:")
print(ADV_FOLDER)


# ============================================================
# IMAGE EXTENSIONS
# ============================================================

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# FUNCTION: LOAD RAW PIXEL FEATURES
# ============================================================

def load_images_from_folder(
    folder,
    label
):

    X = []
    y = []

    folder = Path(folder)

    image_files = [
        file
        for file in folder.iterdir()
        if file.is_file()
        and file.suffix.lower()
        in VALID_EXTENSIONS
    ]

    print(
        f"\nLoading from: {folder}"
    )

    print(
        f"Images found: {len(image_files)}"
    )

    for i, image_path in enumerate(
        image_files
    ):

        try:

            # --------------------------------------------
            # Open image
            # --------------------------------------------

            image = Image.open(
                image_path
            ).convert("RGB")

            # --------------------------------------------
            # Resize to 224 × 224
            # --------------------------------------------

            image = image.resize(
                IMAGE_SIZE
            )

            # --------------------------------------------
            # Convert to numpy
            # --------------------------------------------

            image_array = np.asarray(
                image,
                dtype=np.float32
            )

            # --------------------------------------------
            # Normalize pixels
            # 0-255 → 0-1
            # --------------------------------------------

            image_array = (
                image_array / 255.0
            )

            # --------------------------------------------
            # Flatten
            # 224 × 224 × 3
            # → 150528
            # --------------------------------------------

            feature_vector = (
                image_array.flatten()
            )

            X.append(
                feature_vector
            )

            y.append(
                label
            )

            if (
                (i + 1) % 100 == 0
                or i == len(image_files) - 1
            ):

                print(
                    f"Processed "
                    f"{i + 1}/"
                    f"{len(image_files)}"
                )

        except Exception as e:

            print(
                f"Error loading "
                f"{image_path}: {e}"
            )


    return (
        np.array(X),
        np.array(y)
    )


# ============================================================
# LOAD CLEAN IMAGES
# LABEL = 0
# ============================================================

X_clean, y_clean = load_images_from_folder(
    CLEAN_FOLDER,
    label=0
)


# ============================================================
# LOAD ADVERSARIAL IMAGES
# LABEL = 1
# ============================================================

X_adv, y_adv = load_images_from_folder(
    ADV_FOLDER,
    label=1
)


# ============================================================
# COMBINE DATA
# ============================================================

X = np.concatenate(
    [X_clean, X_adv],
    axis=0
)

y = np.concatenate(
    [y_clean, y_adv],
    axis=0
)


print("\n")
print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(
    "Clean images       :",
    len(X_clean)
)

print(
    "Adversarial images :",
    len(X_adv)
)

print(
    "Total images       :",
    len(X)
)

print(
    "Feature dimension  :",
    X.shape[1]
)

print(
    "Expected dimension :",
    224 * 224 * 3
)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.05,
    random_state=RANDOM_STATE,
    stratify=y
)


print("\n")
print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    "Training images:",
    len(X_train)
)

print(
    "Testing images :",
    len(X_test)
)

print(
    "\nTraining class distribution:"
)

print(
    pd.Series(y_train).value_counts()
)

print(
    "\nTesting class distribution:"
)

print(
    pd.Series(y_test).value_counts()
)


# ============================================================
# STANDARDIZATION
# ============================================================
# Important for KNN because distance is used.
#
# NOTE:
# scaler is fitted ONLY on training data.
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


# ============================================================
# TRAIN KNN
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING KNN")
print("=" * 70)

knn = KNeighborsClassifier(
    n_neighbors=K,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

knn.fit(
    X_train_scaled,
    y_train
)

print(
    "KNN training completed."
)

print(
    "K =", K
)


# ============================================================
# TEST KNN
# ============================================================

print("\n")
print("=" * 70)
print("TESTING KNN")
print("=" * 70)

y_pred = knn.predict(
    X_test_scaled
)

# Probability for ROC-AUC
y_prob = knn.predict_proba(
    X_test_scaled
)[:, 1]


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

TN, FP, FN, TP = cm.ravel()


# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)


# ============================================================
# TPR AND FPR
# ============================================================

# True Positive Rate
# TP / (TP + FN)

TPR = (
    TP /
    (TP + FN)
    if (TP + FN) > 0
    else 0
)


# False Positive Rate
# FP / (FP + TN)

FPR = (
    FP /
    (FP + TN)
    if (FP + TN) > 0
    else 0
)


# ============================================================
# SPECIFICITY
# ============================================================

specificity = (
    TN /
    (TN + FP)
    if (TN + FP) > 0
    else 0
)


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("KNN PRE DETECTOR RESULTS")
print("=" * 70)

print("\nConfusion Matrix:")
print(
    "[[TN, FP],")
print(
    " [FN, TP]]"
)

print(cm)

print("\n")
print("TN =", TN)
print("FP =", FP)
print("FN =", FN)
print("TP =", TP)


print("\n")
print("-" * 70)

print(
    f"Accuracy     : {accuracy:.4f}"
)

print(
    f"Precision    : {precision:.4f}"
)

print(
    f"Recall       : {recall:.4f}"
)

print(
    f"F1 Score     : {f1:.4f}"
)

print(
    f"TPR          : {TPR:.4f}"
)

print(
    f"TPR (%)      : {TPR * 100:.2f}%"
)

print(
    f"FPR          : {FPR:.4f}"
)

print(
    f"FPR (%)      : {FPR * 100:.2f}%"
)

print(
    f"Specificity  : {specificity:.4f}"
)

print(
    f"ROC-AUC      : {roc_auc:.4f}"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Clean",
            "Adversarial"
        ],
        digits=4,
        zero_division=0
    )
)

# *RensNet_ training using each crop one by one***

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --------------------------------------------------
# 1. Check data shape
# --------------------------------------------------
print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)
print("X_val:", x_test.shape)
print("Y_val:", y_test_encoded.shape)

# --------------------------------------------------
# 2. ResNet50
# --------------------------------------------------
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=x_train.shape[1:]
)

# Initially freeze ResNet50 layers
base_model.trainable = True

# --------------------------------------------------
# 3. Classification head
# --------------------------------------------------
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation="softmax")(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

# --------------------------------------------------
# 4. Compile
# --------------------------------------------------
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --------------------------------------------------
# 5. Callbacks
# --------------------------------------------------
checkpoint = ModelCheckpoint(
    "resnet50_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# --------------------------------------------------
# 6. Train for 10 epochs
# --------------------------------------------------
history = model.fit(
    x_train,
    y_train_encoded,
    validation_data=(x_test, y_test_encoded),
    epochs=8,
    batch_size=32,
    callbacks=[checkpoint],
    shuffle=True
)

In [ ]:
# Save final trained model
model.save("resnet50_6class_20epochs.keras")

In [ ]:
import os

print(os.listdir('/kaggle/working'))

# *Image level detection using test data***

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================================
# PATHS
# ============================================================

TEST_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Tomato_test"
)

RESNET_PATH = (
    "/kaggle/working/resnet50_6class_20epochs.keras"
)

IMAGE_SIZE = (224, 224)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# LOAD TRAINED RESNET50
# ============================================================

print("Loading ResNet50...")

resnet = tf.keras.models.load_model(
    RESNET_PATH
)

print("ResNet50 loaded successfully.")


# ============================================================
# GET CLASS NAMES
# ============================================================

class_folders = sorted(
    [
        folder
        for folder in TEST_PATH.iterdir()
        if folder.is_dir()
    ],
    key=lambda x: x.name
)

class_names = [
    folder.name
    for folder in class_folders
]

print("\nTest classes:")

for i, name in enumerate(class_names):
    print(i, "->", name)


# ============================================================
# IMPORTANT
# ============================================================
# Your ResNet was trained with n classes.
# The class order used here MUST be the SAME order used
# during ResNet training.
#
# If your training used class_names from a specific
# ImageDataGenerator / directory order, use that exact order.
# ============================================================


# ============================================================
# COUNTERS
# ============================================================

total_images = 0

clean_detected = 0
adversarial_detected = 0

cnn_true_labels = []
cnn_pred_labels = []


# ============================================================
# STORE INFORMATION FOR EACH CLEAN-PASSED IMAGE
# ============================================================

clean_passed_images = []


# ============================================================
# PROCESS TEST DATA
# ============================================================

print("\n")
print("=" * 70)
print("STARTING KNN → RESNET TESTING")
print("=" * 70)


for true_class_index, class_folder in enumerate(
    class_folders
):

    class_name = class_folder.name

    image_files = sorted(
        [
            file
            for file in class_folder.iterdir()
            if file.is_file()
            and file.suffix.lower()
            in VALID_EXTENSIONS
        ]
    )


    print(
        f"\nClass: {class_name}"
    )

    print(
        f"Images: {len(image_files)}"
    )


    for image_path in image_files:

        total_images += 1


        # ====================================================
        # LOAD IMAGE
        # ====================================================

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            print(
                f"Error loading {image_path}: {e}"
            )

            continue


        # ====================================================
        # SAME PREPROCESSING AS KNN TRAINING
        # ====================================================

        image = image.resize(
            IMAGE_SIZE
        )

        image_array = np.asarray(
            image,
            dtype=np.float32
        )

        # Same normalization
        image_array = (
            image_array / 255.0
        )

        # Same flattening
        feature_vector = (
            image_array.flatten()
        )


        # Shape:
        # (150528,)
        # Convert to:
        # (1, 150528)

        feature_vector = (
            feature_vector.reshape(1, -1)
        )


        # ====================================================
        # SAME TRAINED SCALER
        # ====================================================

        feature_scaled = scaler.transform(
            feature_vector
        )


        # ====================================================
        # EXISTING TRAINED KNN
        # ====================================================

        knn_prediction = knn.predict(
            feature_scaled
        )[0]


        # ====================================================
        # KNN:
        #
        # 0 = Clean
        # 1 = Adversarial
        # ====================================================

        if knn_prediction == 1:

            # ----------------------------------------------
            # ADVERSARIAL
            # ----------------------------------------------

            adversarial_detected += 1

            # Do NOT send to CNN
            continue


        else:

            # ----------------------------------------------
            # CLEAN
            # ----------------------------------------------

            clean_detected += 1


        # ====================================================
        # SEND CLEAN-DETECTED IMAGE TO RESNET50
        # ====================================================

        cnn_image = np.asarray(
            image,
            dtype=np.float32
        )

        # ResNet gets image shape:
        # (1, 224, 224, 3)

        cnn_input = (
            np.expand_dims(
                cnn_image,
                axis=0
            )
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # Use SAME preprocessing as ResNet training.
        #
        # If your ResNet was trained using /255.0,
        # use /255.0.
        #
        # If it used another preprocessing function,
        # replace this part accordingly.
        # ----------------------------------------------------

        cnn_input = (
            cnn_input / 255.0
        )


        # ====================================================
        # RESNET PREDICTION
        # ====================================================

        prediction = resnet.predict(
            cnn_input,
            verbose=0
        )


        predicted_class = int(
            np.argmax(
                prediction,
                axis=1
            )[0]
        )


        # ====================================================
        # STORE TRUE + PREDICTED CLASS
        # ====================================================

        cnn_true_labels.append(
            true_class_index
        )

        cnn_pred_labels.append(
            predicted_class
        )


        clean_passed_images.append({

            "image": str(image_path),

            "true_class": class_name,

            "true_class_index":
                true_class_index,

            "knn_prediction":
                "Clean",

            "cnn_prediction":
                class_names[predicted_class]

        })


# ============================================================
# RESULTS: DETECTOR ROUTING
# ============================================================

print("\n")
print("=" * 70)
print("KNN DETECTOR ROUTING RESULTS")
print("=" * 70)

print(
    f"Total test images       : {total_images}"
)

print(
    f"Detected as Clean       : {clean_detected}"
)

print(
    f"Detected as Adversarial : {adversarial_detected}"
)


# ============================================================
# CNN RESULTS ON CLEAN-DETECTED IMAGES
# ============================================================

print("\n")
print("=" * 70)
print("RESNET50 RESULTS")
print("=" * 70)


if len(cnn_true_labels) == 0:

    print(
        "No images were classified as Clean "
        "by KNN. ResNet was not evaluated."
    )

else:

    cnn_accuracy = accuracy_score(
        cnn_true_labels,
        cnn_pred_labels
    )

    print(
        f"Images passed to CNN : "
        f"{len(cnn_true_labels)}"
    )

    print(
        f"CNN Accuracy         : "
        f"{cnn_accuracy:.4f}"
    )

    print(
        f"CNN Accuracy (%)     : "
        f"{cnn_accuracy * 100:.2f}%"
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    print("\n")
    print("=" * 70)
    print("CNN CLASSIFICATION REPORT")
    print("=" * 70)

    print(
        classification_report(
            cnn_true_labels,
            cnn_pred_labels,
            labels=list(
                range(len(class_names))
            ),
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        cnn_true_labels,
        cnn_pred_labels,
        labels=list(
            range(len(class_names))
        )
    )

    print("\n")
    print("=" * 70)
    print("CNN CONFUSION MATRIX")
    print("=" * 70)

    print(cm)


# ============================================================
# SAVE CLEAN-PASSED IMAGE INFORMATION
# ============================================================

import pandas as pd

results_df = pd.DataFrame(
    clean_passed_images
)

results_path = (
    "/kaggle/working/"
    "clean_passed_to_resnet.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("\n")
print("=" * 70)
print("IMAGE-LEVEL RESULTS SAVED")
print("=" * 70)

print(
    results_path
)

# Deep feature level KNN training 

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import tensorflow as tf

from tensorflow.keras.models import Model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)


# ============================================================
# PATHS
# ============================================================

DATASET_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/combine-tomato"
)

CLEAN_FOLDER = DATASET_PATH / "Clean"
ADV_FOLDER = DATASET_PATH / "Adversarial"

RESNET_PATH = (
    "/kaggle/working/resnet50_6class_20epochs.keras"
)

IMAGE_SIZE = (224, 224)

RANDOM_STATE = 42

K = 5

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# LOAD TRAINED RESNET50
# ============================================================

print("=" * 70)
print("LOADING RESNET50")
print("=" * 70)

resnet = tf.keras.models.load_model(
    RESNET_PATH
)

print("ResNet50 loaded successfully.")


# ============================================================
# CHECK MODEL
# ============================================================

print("\nResNet input shape:")
print(resnet.input_shape)

print("\nResNet output shape:")
print(resnet.output_shape)


# ============================================================
# FEATURE EXTRACTION LAYER
# ============================================================
#
# IMPORTANT:
# We use the layer immediately before the final
# 6-class classification layer.
#
# If your model ends like:
#
# GlobalAveragePooling2D
#          ↓
# Dense(6)
#
# then this gives the 2048-dimensional feature vector.
#
# ============================================================

feature_layer = None

for layer in reversed(resnet.layers):

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        feature_layer = layer
        break


if feature_layer is None:

    raise ValueError(
        "GlobalAveragePooling2D layer not found."
    )


feature_extractor = Model(
    inputs=resnet.input,
    outputs=feature_layer.output
)


print("\nFeature extraction layer:")
print(feature_layer.name)

print(
    "Feature output shape:",
    feature_extractor.output_shape
)


# ============================================================
# FUNCTION: GET IMAGES
# ============================================================

def get_images(folder):

    folder = Path(folder)

    if not folder.exists():
        return []

    return sorted(
        [
            file
            for file in folder.iterdir()
            if file.is_file()
            and file.suffix.lower()
            in VALID_EXTENSIONS
        ]
    )


# ============================================================
# FUNCTION: EXTRACT RESNET FEATURES
# ============================================================

def extract_features(
    folder,
    label
):

    images = get_images(folder)

    features = []
    labels = []

    print("\n")
    print("-" * 70)
    print("Folder:", folder)
    print("Images:", len(images))
    print("Label :", label)
    print("-" * 70)


    for i, image_path in enumerate(
        images
    ):

        try:

            # ------------------------------------------------
            # Load image
            # ------------------------------------------------

            image = Image.open(
                image_path
            ).convert("RGB")


            # ------------------------------------------------
            # Resize
            # ------------------------------------------------

            image = image.resize(
                IMAGE_SIZE
            )


            # ------------------------------------------------
            # Convert to numpy
            # ------------------------------------------------

            image_array = np.asarray(
                image,
                dtype=np.float32
            )


            # ------------------------------------------------
            # Same /255 preprocessing
            # used by the ResNet training
            # ------------------------------------------------

            image_array = (
                image_array / 255.0
            )


            # ------------------------------------------------
            # Add batch dimension
            # ------------------------------------------------

            image_array = np.expand_dims(
                image_array,
                axis=0
            )


            # ------------------------------------------------
            # Extract deep feature
            # ------------------------------------------------

            feature = (
                feature_extractor.predict(
                    image_array,
                    verbose=0
                )
            )


            # ------------------------------------------------
            # Flatten feature if required
            # ------------------------------------------------

            feature = feature.flatten()


            features.append(
                feature
            )

            labels.append(
                label
            )


            if (
                (i + 1) % 50 == 0
                or i == len(images) - 1
            ):

                print(
                    f"Processed "
                    f"{i + 1}/"
                    f"{len(images)}"
                )


        except Exception as e:

            print(
                f"Error processing "
                f"{image_path}: {e}"
            )


    return (
        np.array(features),
        np.array(labels)
    )


# ============================================================
# EXTRACT CLEAN FEATURES
# LABEL = 0
# ============================================================

X_clean, y_clean = extract_features(
    CLEAN_FOLDER,
    label=0
)


# ============================================================
# EXTRACT ADVERSARIAL FEATURES
# LABEL = 1
# ============================================================

X_adv, y_adv = extract_features(
    ADV_FOLDER,
    label=1
)


# ============================================================
# COMBINE FEATURES
# ============================================================

X = np.concatenate(
    [
        X_clean,
        X_adv
    ],
    axis=0
)

y = np.concatenate(
    [
        y_clean,
        y_adv
    ],
    axis=0
)


# ============================================================
# DATASET INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("POST DETECTOR DATASET")
print("=" * 70)

print(
    "Clean images       :",
    len(X_clean)
)

print(
    "Adversarial images :",
    len(X_adv)
)

print(
    "Total images       :",
    len(X)
)

print(
    "Feature dimension  :",
    X.shape[1]
)


# ============================================================
# EXPECTED FEATURE DIMENSION
# ============================================================

print("\n")
print(
    "Expected ResNet feature dimension "
    "if GAP is used: 2048"
)


# ============================================================
# STANDARDIZATION
# ============================================================
#
# Same principle as PRE:
# KNN is distance-based.
#
# IMPORTANT:
# Scaler is fitted on the complete detector-training
# feature dataset because this code trains the POST
# detector using the already-created detector dataset.
# ============================================================

scaler_post = StandardScaler()

X_scaled = scaler_post.fit_transform(
    X
)


# ============================================================
# TRAIN POST KNN
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING POST KNN")
print("=" * 70)

knn_post = KNeighborsClassifier(
    n_neighbors=K,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

knn_post.fit(
    X_scaled,
    y
)

print(
    "POST KNN training completed."
)

print(
    "K =", K
)


# ============================================================
# TRAINING-SET EVALUATION
# ============================================================
#
# This is only a sanity check.
# The real POST performance must be evaluated
# on an independent test dataset.
# ============================================================

y_train_pred = knn_post.predict(
    X_scaled
)

y_train_prob = knn_post.predict_proba(
    X_scaled
)[:, 1]


cm_train = confusion_matrix(
    y,
    y_train_pred,
    labels=[0, 1]
)

TN, FP, FN, TP = cm_train.ravel()


accuracy = accuracy_score(
    y,
    y_train_pred
)

precision = precision_score(
    y,
    y_train_pred,
    zero_division=0
)

recall = recall_score(
    y,
    y_train_pred,
    zero_division=0
)

f1 = f1_score(
    y,
    y_train_pred,
    zero_division=0
)

tpr = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)

fpr = (
    FP / (FP + TN)
    if (FP + TN) > 0
    else 0
)

roc_auc = roc_auc_score(
    y,
    y_train_prob
)


print("\n")
print("=" * 70)
print("POST KNN TRAINING-SET SANITY CHECK")
print("=" * 70)

print("\nConfusion Matrix:")
print(cm_train)

print("\nTN:", TN)
print("FP:", FP)
print("FN:", FN)
print("TP:", TP)

print("\nAccuracy :", f"{accuracy:.4f}")
print("Precision:", f"{precision:.4f}")
print("Recall   :", f"{recall:.4f}")
print("F1 Score :", f"{f1:.4f}")
print("TPR      :", f"{tpr:.4f}")
print("FPR      :", f"{fpr:.4f}")
print("ROC-AUC  :", f"{roc_auc:.4f}")


# ============================================================
# SAVE POST KNN + SCALER
# ============================================================

import joblib

knn_path = (
    "/kaggle/working/"
    "post_knn_resnet_features.pkl"
)

scaler_path = (
    "/kaggle/working/"
    "post_knn_scaler.pkl"
)

joblib.dump(
    knn_post,
    knn_path
)

joblib.dump(
    scaler_post,
    scaler_path
)


print("\n")
print("=" * 70)
print("POST MODEL SAVED")
print("=" * 70)

print(
    "KNN:",
    knn_path
)

print(
    "Scaler:",
    scaler_path
)


# ============================================================
# SAVE TRAINING FEATURES
# ============================================================

np.save(
    "/kaggle/working/post_features.npy",
    X
)

np.save(
    "/kaggle/working/post_labels.npy",
    y
)

print("\nFeature files saved.")

# *Feature base testing using knn***

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import tensorflow as tf

from tensorflow.keras.models import Model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# PATHS
# ============================================================

TEST_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Tomato_test"
)

RESNET_PATH = (
    "/kaggle/working/resnet50_6class_20epochs.keras"
)

IMAGE_SIZE = (224, 224)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# CHECK EXISTING TRAINED OBJECTS
# ============================================================

if "knn_post" not in globals():

    raise NameError(
        "knn_post is not available. "
        "Run your POST-KNN training cell first."
    )


if "scaler_post" not in globals():

    raise NameError(
        "scaler_post is not available. "
        "Run your POST-KNN training cell first."
    )


if "label_encoder" not in globals():

    raise NameError(
        "label_encoder is not available. "
        "Run the LabelEncoder cell used during CNN training."
    )


# ============================================================
# PRINT LABEL ENCODER MAPPING
# ============================================================

print("=" * 70)
print("LABEL ENCODER MAPPING USED BY CNN")
print("=" * 70)

for index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{class_name}  -->  {index}"
    )


# ============================================================
# LOAD SAVED RESNET50
# ============================================================

print("\n")
print("=" * 70)
print("LOADING RESNET50")
print("=" * 70)

resnet = tf.keras.models.load_model(
    RESNET_PATH
)

print(
    "ResNet50 loaded successfully."
)


# ============================================================
# CREATE FEATURE EXTRACTOR
# ============================================================
#
# SAME FEATURE LAYER USED DURING POST-KNN TRAINING
# ============================================================

feature_layer = None


for layer in reversed(
    resnet.layers
):

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):

        feature_layer = layer

        break


if feature_layer is None:

    raise ValueError(
        "GlobalAveragePooling2D layer not found."
    )


feature_extractor = Model(
    inputs=resnet.input,
    outputs=feature_layer.output
)


print("\n")
print("=" * 70)
print("FEATURE EXTRACTOR")
print("=" * 70)

print(
    "Feature layer:",
    feature_layer.name
)

print(
    "Feature output shape:",
    feature_extractor.output_shape
)


# ============================================================
# GET TEST CLASS FOLDERS
# ============================================================

class_folders = sorted(
    [
        folder
        for folder in TEST_PATH.iterdir()
        if folder.is_dir()
    ],
    key=lambda x: x.name
)


print("\n")
print("=" * 70)
print("TEST DATASET CLASSES")
print("=" * 70)


for folder in class_folders:

    class_name = folder.name

    # --------------------------------------------------------
    # IMPORTANT:
    # Use SAME LabelEncoder used during CNN training.
    # --------------------------------------------------------

    try:

        encoded_label = int(
            label_encoder.transform(
                [class_name]
            )[0]
        )

    except ValueError:

        raise ValueError(
            f"Class '{class_name}' "
            f"was not present in the LabelEncoder "
            f"used during CNN training."
        )


    print(
        f"{class_name} --> encoded label {encoded_label}"
    )


# ============================================================
# RESULT STORAGE
# ============================================================

total_images = 0

adversarial_count = 0

clean_count = 0

clean_true_labels = []

clean_pred_labels = []

results = []


# ============================================================
# PROCESS TEST DATA
# ============================================================

print("\n")
print("=" * 70)
print("STARTING POST DETECTOR TESTING")
print("=" * 70)


for class_folder in class_folders:

    # ========================================================
    # ORIGINAL DISEASE CLASS NAME
    # ========================================================

    true_class_name = class_folder.name


    # ========================================================
    # GET EXACT ENCODED LABEL USING TRAINING ENCODER
    # ========================================================

    true_class_index = int(
        label_encoder.transform(
            [true_class_name]
        )[0]
    )


    # ========================================================
    # GET IMAGES
    # ========================================================

    image_files = sorted(
        [
            file
            for file in class_folder.iterdir()
            if file.is_file()
            and file.suffix.lower()
            in VALID_EXTENSIONS
        ]
    )


    print("\n")
    print(
        f"Class: {true_class_name}"
    )

    print(
        f"Encoded label: {true_class_index}"
    )

    print(
        f"Images: {len(image_files)}"
    )


    # ========================================================
    # PROCESS EACH IMAGE
    # ========================================================

    for image_path in image_files:

        total_images += 1


        # ====================================================
        # LOAD IMAGE
        # ====================================================

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            print(
                f"Error loading: "
                f"{image_path}"
            )

            print(e)

            continue


        # ====================================================
        # RESIZE
        # ====================================================

        image = image.resize(
            IMAGE_SIZE
        )


        # ====================================================
        # SAME PREPROCESSING USED DURING POST TRAINING
        # ====================================================

        image_array = np.asarray(
            image,
            dtype=np.float32
        )


        image_array = (
            image_array / 255.0
        )


        # ====================================================
        # ADD BATCH DIMENSION
        # ====================================================

        batch = np.expand_dims(
            image_array,
            axis=0
        )


        # ====================================================
        # RESNET FEATURE EXTRACTION
        # ====================================================

        feature = feature_extractor.predict(
            batch,
            verbose=0
        )


        # ====================================================
        # FLATTEN FEATURE
        # ====================================================

        feature = feature.flatten()


        feature = feature.reshape(
            1,
            -1
        )


        # ====================================================
        # SAME POST SCALER
        # ====================================================

        feature_scaled = (
            scaler_post.transform(
                feature
            )
        )


        # ====================================================
        # EXISTING TRAINED POST-KNN
        # ====================================================

        knn_prediction = int(
            knn_post.predict(
                feature_scaled
            )[0]
        )


        # ====================================================
        # KNN = ADVERSARIAL
        # ====================================================

        if knn_prediction == 1:

            adversarial_count += 1


            results.append({

                "image":
                    str(image_path),

                "true_class":
                    true_class_name,

                "true_class_index":
                    true_class_index,

                "KNN_prediction":
                    "Adversarial",

                "CNN_prediction":
                    "Not Tested"

            })


            # ------------------------------------------------
            # IMPORTANT:
            # Adversarial image is removed from
            # downstream CNN classification.
            # ------------------------------------------------

            continue


        # ====================================================
        # KNN = CLEAN
        # ====================================================

        clean_count += 1


        # ====================================================
        # SEND CLEAN IMAGE TO ORIGINAL RESNET50
        # ====================================================

        cnn_output = resnet.predict(
            batch,
            verbose=0
        )


        # ====================================================
        # CNN OUTPUT -> ENCODED CLASS
        # ====================================================

        predicted_class_index = int(
            np.argmax(
                cnn_output,
                axis=1
            )[0]
        )


        # ====================================================
        # STORE TRUE/PREDICTED ENCODED LABELS
        # ====================================================

        clean_true_labels.append(
            true_class_index
        )

        clean_pred_labels.append(
            predicted_class_index
        )


        # ====================================================
        # CONVERT PREDICTED LABEL BACK TO CLASS NAME
        # ====================================================

        predicted_class_name = (
            label_encoder.inverse_transform(
                [predicted_class_index]
            )[0]
        )


        # ====================================================
        # SAVE IMAGE LEVEL RESULT
        # ====================================================

        results.append({

            "image":
                str(image_path),

            "true_class":
                true_class_name,

            "true_class_index":
                true_class_index,

            "KNN_prediction":
                "Clean",

            "CNN_prediction":
                predicted_class_name,

            "CNN_prediction_index":
                predicted_class_index

        })


# ============================================================
# POST-KNN ROUTING RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("POST-KNN DETECTOR RESULTS")
print("=" * 70)

print(
    f"Total test images       : "
    f"{total_images}"
)

print(
    f"Adversarial detected    : "
    f"{adversarial_count}"
)

print(
    f"Clean passed to CNN     : "
    f"{clean_count}"
)

print(
    f"Total checked           : "
    f"{adversarial_count + clean_count}"
)


# ============================================================
# PERCENTAGES
# ============================================================

if total_images > 0:

    print(
        f"\nAdversarial detected (%) : "
        f"{adversarial_count / total_images * 100:.2f}%"
    )

    print(
        f"Clean passed (%)         : "
        f"{clean_count / total_images * 100:.2f}%"
    )


# ============================================================
# CNN CLASSIFICATION RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("RESNET50 CLASSIFICATION AFTER POST DETECTION")
print("=" * 70)


if len(clean_true_labels) == 0:

    print(
        "No Clean images were passed "
        "to the CNN."
    )

else:

    cnn_accuracy = accuracy_score(
        clean_true_labels,
        clean_pred_labels
    )


    print(
        f"Images passed to CNN : "
        f"{len(clean_true_labels)}"
    )

    print(
        f"CNN Accuracy         : "
        f"{cnn_accuracy:.4f}"
    )

    print(
        f"CNN Accuracy (%)     : "
        f"{cnn_accuracy * 100:.2f}%"
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    print("\n")
    print("=" * 70)
    print("CNN CLASSIFICATION REPORT")
    print("=" * 70)


    print(
        classification_report(
            clean_true_labels,
            clean_pred_labels,
            labels=list(
                range(
                    len(
                        label_encoder.classes_
                    )
                )
            ),
            target_names=(
                label_encoder.classes_
            ),
            digits=4,
            zero_division=0
        )
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cnn_cm = confusion_matrix(
        clean_true_labels,
        clean_pred_labels,
        labels=list(
            range(
                len(
                    label_encoder.classes_
                )
            )
        )
    )


    print("\n")
    print("=" * 70)
    print("CNN CONFUSION MATRIX")
    print("=" * 70)

    print(cnn_cm)


# ============================================================
# SAVE IMAGE-LEVEL RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)


results_path = (
    "/kaggle/working/"
    "POST_detector_test_results.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    "Total test images      :",
    total_images
)

print(
    "Adversarial removed    :",
    adversarial_count
)

print(
    "Clean sent to CNN      :",
    clean_count
)

print(
    "\nDetailed results saved:"
)

print(
    results_path
)

In [ ]:
print("ResNet output:", resnet.output_shape)
print("CNN classes:", len(label_encoder.classes_))
print("CNN class names:")
print(label_encoder.classes_)

# *pre_result in all data set created using different epsilon***

In [ ]:
# ============================================================
# PRE-MODEL ADVERSARIAL EXAMPLE DETECTION
# KNN TRAINING + MIXED DATASET EVALUATION + RESNET50
# ============================================================

import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

from tensorflow.keras.models import load_model


# ============================================================
# 1. PATHS
# ============================================================

# ------------------------------------------------------------
# Separate dataset ONLY for KNN training
#
# Expected structure:
#
# KNN_TRAIN_PATH/
# ├── Clean/
# │    ├── image1.jpg
# │    ├── image2.jpg
# │    └── ...
# │
# └── Adversarial/
#      ├── image1.jpg
#      ├── image2.jpg
#      └── ...
# ------------------------------------------------------------

KNN_TRAIN_PATH = "/kaggle/input/datasets/yadavmohit2000/combine-tomato"


# ------------------------------------------------------------
# Mixed test dataset
#
# Expected structure:
#
# TEST_PATH/
# ├── Disease_Class_1/
# │    ├── first 30 images = Clean
# │    └── next 10 images  = Adversarial
# │
# ├── Disease_Class_2/
# │    ├── first 30 images = Clean
# │    └── next 10 images  = Adversarial
# │
# └── ...
# ------------------------------------------------------------

TEST_PATH = "/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Tomato_test"


# ------------------------------------------------------------
# Trained ResNet50 disease-classification model
# ------------------------------------------------------------

RESNET_PATH = "/kaggle/working/resnet50_6class_20epochs.keras"


# ============================================================
# 2. PARAMETERS
# ============================================================

IMAGE_SIZE = (224, 224)

KNN_K = 5

RANDOM_STATE = 42

# Number of clean/adversarial images in each disease class
CLEAN_PER_CLASS = 30
ADV_PER_CLASS = 10


# ============================================================
# 3. IMAGE PREPROCESSING FOR PRE-MODEL KNN
# ============================================================

def preprocess_for_pre_knn(image_path):

    img = Image.open(image_path).convert("RGB")

    # Resize to 224 x 224
    img = img.resize(IMAGE_SIZE)

    # Convert to NumPy array
    img = np.array(img, dtype=np.float32)

    # Normalize 0-255 --> 0-1
    img = img / 255.0

    # Flatten:
    # 224 x 224 x 3 = 150528 features
    img = img.flatten()

    return img


# ============================================================
# 4. LOAD KNN TRAINING DATASET
# ============================================================

print("\n===================================================")
print("LOADING KNN TRAINING DATASET")
print("===================================================")

knn_X = []
knn_y = []

class_mapping = {
    "Clean": 0,
    "Adversarial": 1
}

for class_name, class_label in class_mapping.items():

    class_path = os.path.join(KNN_TRAIN_PATH, class_name)

    if not os.path.isdir(class_path):
        print(f"WARNING: Folder not found: {class_path}")
        continue

    image_files = []

    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        image_files.extend(glob.glob(os.path.join(class_path, ext)))

    image_files = sorted(image_files)

    print(f"{class_name}: {len(image_files)} images")

    for image_path in image_files:

        try:

            features = preprocess_for_pre_knn(image_path)

            knn_X.append(features)
            knn_y.append(class_label)

        except Exception as e:

            print(f"Error processing {image_path}: {e}")


knn_X = np.array(knn_X, dtype=np.float32)
knn_y = np.array(knn_y, dtype=np.int32)


print("\nKNN training data shape:", knn_X.shape)
print("KNN labels shape:", knn_y.shape)

print("\nTraining class distribution:")

unique_labels, counts = np.unique(knn_y, return_counts=True)

for label, count in zip(unique_labels, counts):

    if label == 0:
        print(f"Clean       : {count}")
    else:
        print(f"Adversarial : {count}")


# ============================================================
# 5. STANDARD SCALER
# ============================================================

print("\n===================================================")
print("FITTING STANDARD SCALER")
print("===================================================")

scaler_pre = StandardScaler()

knn_X_scaled = scaler_pre.fit_transform(knn_X)

print("Scaled training data shape:", knn_X_scaled.shape)


# ============================================================
# 6. TRAIN PRE-KNN
# ============================================================

print("\n===================================================")
print("TRAINING PRE-KNN")
print("===================================================")

knn_pre = KNeighborsClassifier(
    n_neighbors=KNN_K,
    metric="euclidean",
    weights="distance"
)

knn_pre.fit(knn_X_scaled, knn_y)

print("Pre-KNN training completed.")


# ============================================================
# 7. SAVE TRAINED KNN + SCALER
# ============================================================

KNN_SAVE_PATH = "/kaggle/working/knn_pre.pkl"
SCALER_SAVE_PATH = "/kaggle/working/scaler_pre.pkl"

joblib.dump(knn_pre, KNN_SAVE_PATH)
joblib.dump(scaler_pre, SCALER_SAVE_PATH)

print("\nSaved:")
print(KNN_SAVE_PATH)
print(SCALER_SAVE_PATH)


# ============================================================
# 8. LOAD RESNET50
# ============================================================

print("\n===================================================")
print("LOADING RESNET50")
print("===================================================")

resnet_model = load_model(RESNET_PATH)

print("ResNet50 loaded successfully.")


# ============================================================
# 9. FIND DISEASE CLASS FOLDERS
# ============================================================

disease_classes = sorted([
    folder
    for folder in os.listdir(TEST_PATH)
    if os.path.isdir(os.path.join(TEST_PATH, folder))
])

print("\nDisease classes found:")

for cls in disease_classes:
    print(" -", cls)


# ============================================================
# 10. LABEL ENCODER
# ============================================================

label_encoder = LabelEncoder()

label_encoder.fit(disease_classes)

print("\nLabel mapping:")

for i, cls in enumerate(label_encoder.classes_):
    print(i, "=", cls)


# ============================================================
# 11. STORAGE FOR RESULTS
# ============================================================

all_results = []

y_true_detection = []
y_pred_detection = []
y_score_detection = []

cnn_true = []
cnn_pred = []


# ============================================================
# 12. PROCESS MIXED TEST DATASET
# ============================================================

print("\n===================================================")
print("STARTING PRE-KNN EVALUATION")
print("===================================================")

for disease_class in disease_classes:

    class_path = os.path.join(TEST_PATH, disease_class)

    image_files = []

    for ext in ["*.jpg", "*.jpeg", "*.png",
                "*.JPG", "*.JPEG", "*.PNG"]:

        image_files.extend(
            glob.glob(os.path.join(class_path, ext))
        )

    image_files = sorted(image_files)

    print("\n---------------------------------------------")
    print("Class:", disease_class)
    print("Total images:", len(image_files))
    print("---------------------------------------------")

    # --------------------------------------------------------
    # Check exactly 40 images
    # --------------------------------------------------------

    if len(image_files) != CLEAN_PER_CLASS + ADV_PER_CLASS:

        print(
            f"WARNING: {disease_class} contains "
            f"{len(image_files)} images instead of 40."
        )

    # --------------------------------------------------------
    # First 30 = Clean
    # Next 10 = Adversarial
    # --------------------------------------------------------

    for index, image_path in enumerate(image_files):

        # Ground truth
        if index < CLEAN_PER_CLASS:

            true_detection_label = 0
            true_detection_name = "Clean"

        else:

            true_detection_label = 1
            true_detection_name = "Adversarial"


        # ----------------------------------------------------
        # PREPROCESS IMAGE
        # ----------------------------------------------------

        try:

            features = preprocess_for_pre_knn(image_path)

            features = features.reshape(1, -1)

            # IMPORTANT:
            # Only TRANSFORM using scaler trained on
            # separate KNN training dataset.
            scaled_features = scaler_pre.transform(features)


            # ------------------------------------------------
            # KNN PREDICTION
            # ------------------------------------------------

            knn_prediction = knn_pre.predict(
                scaled_features
            )[0]

            # Probability/score for Adversarial class
            probability = knn_pre.predict_proba(
                scaled_features
            )[0]

            # Locate probability corresponding to class 1
            class_indices = list(knn_pre.classes_)

            if 1 in class_indices:

                adv_index = class_indices.index(1)

                adversarial_score = probability[adv_index]

            else:

                adversarial_score = 0.0


            # ------------------------------------------------
            # PREDICTED DETECTION LABEL
            # ------------------------------------------------

            predicted_detection_label = int(knn_prediction)

            if predicted_detection_label == 1:

                predicted_detection_name = "Adversarial"

            else:

                predicted_detection_name = "Clean"


            # ------------------------------------------------
            # STORE DETECTION RESULTS
            # ------------------------------------------------

            y_true_detection.append(
                true_detection_label
            )

            y_pred_detection.append(
                predicted_detection_label
            )

            y_score_detection.append(
                adversarial_score
            )


            # ------------------------------------------------
            # CNN CLASSIFICATION
            #
            # Only KNN-clean images are passed to ResNet50
            # ------------------------------------------------

            cnn_prediction_label = None
            cnn_prediction_name = None

            if predicted_detection_label == 0:

                img = Image.open(image_path).convert("RGB")

                img = img.resize(IMAGE_SIZE)

                img_array = np.array(
                    img,
                    dtype=np.float32
                )

                img_array = img_array / 255.0

                img_array = np.expand_dims(
                    img_array,
                    axis=0
                )

                # ResNet prediction
                cnn_output = resnet_model.predict(
                    img_array,
                    verbose=0
                )

                cnn_prediction_label = int(
                    np.argmax(cnn_output[0])
                )

                # Convert predicted number to disease name
                if cnn_prediction_label < len(
                    label_encoder.classes_
                ):

                    cnn_prediction_name = (
                        label_encoder.inverse_transform(
                            [cnn_prediction_label]
                        )[0]
                    )

                else:

                    cnn_prediction_name = str(
                        cnn_prediction_label
                    )


                # True disease label
                true_cnn_label = label_encoder.transform(
                    [disease_class]
                )[0]

                cnn_true.append(true_cnn_label)
                cnn_pred.append(cnn_prediction_label)


            # ------------------------------------------------
            # SAVE IMAGE LEVEL RESULT
            # ------------------------------------------------

            all_results.append({

                "Disease_Class": disease_class,

                "Image_Name": os.path.basename(
                    image_path
                ),

                "Image_Index": index + 1,

                "Ground_Truth_Detection":
                    true_detection_name,

                "Ground_Truth_Detection_Label":
                    true_detection_label,

                "KNN_Prediction":
                    predicted_detection_name,

                "KNN_Prediction_Label":
                    predicted_detection_label,

                "Adversarial_Score":
                    adversarial_score,

                "CNN_Prediction":
                    cnn_prediction_name

            })


        except Exception as e:

            print(
                f"Error processing "
                f"{image_path}: {e}"
            )


# ============================================================
# 13. CONVERT RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)

print("\n===================================================")
print("IMAGE LEVEL RESULTS")
print("===================================================")

print(results_df.head())


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

y_true_detection = np.array(
    y_true_detection
)

y_pred_detection = np.array(
    y_pred_detection
)

y_score_detection = np.array(
    y_score_detection
)


cm = confusion_matrix(
    y_true_detection,
    y_pred_detection,
    labels=[0, 1]
)

TN, FP, FN, TP = cm.ravel()


print("\n===================================================")
print("KNN CONFUSION MATRIX")
print("===================================================")

print(cm)

print("\nTN =", TN)
print("FP =", FP)
print("FN =", FN)
print("TP =", TP)


# ============================================================
# 15. KNN DETECTION METRICS
# ============================================================

total_images = len(y_true_detection)

clean_images = np.sum(
    y_true_detection == 0
)

adversarial_images = np.sum(
    y_true_detection == 1
)

adversarial_detected = TP

clean_passed = TN

accuracy = accuracy_score(
    y_true_detection,
    y_pred_detection
)

precision = precision_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)

recall = recall_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)

f1 = f1_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)

# False Negative Rate
if (TP + FN) > 0:

    fnr = FN / (TP + FN)

else:

    fnr = 0.0


# False Positive Rate
if (FP + TN) > 0:

    fpr = FP / (FP + TN)

else:

    fpr = 0.0


# ============================================================
# 16. ROC + AUC
# ============================================================

if len(np.unique(y_true_detection)) == 2:

    fpr_curve, tpr_curve, thresholds = roc_curve(
        y_true_detection,
        y_score_detection
    )

    auc_value = roc_auc_score(
        y_true_detection,
        y_score_detection
    )

else:

    fpr_curve = np.array([])
    tpr_curve = np.array([])
    thresholds = np.array([])
    auc_value = np.nan


# ============================================================
# 17. PRINT KNN RESULTS
# ============================================================

print("\n===================================================")
print("PRE-KNN DETECTION RESULTS")
print("===================================================")

print(f"Total Images          : {total_images}")

print(f"Clean Images          : {clean_images}")

print(f"Adversarial Images    : {adversarial_images}")

print(f"Adversarial Detected  : {adversarial_detected}")

print(f"Clean Passed          : {clean_passed}")

print(f"TP                    : {TP}")

print(f"TN                    : {TN}")

print(f"FP                    : {FP}")

print(f"FN                    : {FN}")

print(f"Accuracy              : {accuracy * 100:.2f}%")

print(f"Precision             : {precision * 100:.2f}%")

print(f"Recall / TPR          : {recall * 100:.2f}%")

print(f"FNR                   : {fnr * 100:.2f}%")

print(f"FPR                   : {fpr * 100:.2f}%")

print(f"F1 Score              : {f1 * 100:.2f}%")

print(f"ROC-AUC               : {auc_value:.4f}")


# ============================================================
# 18. CNN METRICS
# ============================================================

print("\n===================================================")
print("RESNET50 CNN RESULTS")
print("===================================================")

cnn_true = np.array(cnn_true)
cnn_pred = np.array(cnn_pred)


if len(cnn_true) > 0:

    cnn_accuracy = accuracy_score(
        cnn_true,
        cnn_pred
    )

    cnn_precision = precision_score(
        cnn_true,
        cnn_pred,
        average="weighted",
        zero_division=0
    )

    cnn_recall = recall_score(
        cnn_true,
        cnn_pred,
        average="weighted",
        zero_division=0
    )

    cnn_f1 = f1_score(
        cnn_true,
        cnn_pred,
        average="weighted",
        zero_division=0
    )


    print(
        f"Clean-passed images sent to CNN : "
        f"{len(cnn_true)}"
    )

    print(
        f"CNN Accuracy                    : "
        f"{cnn_accuracy * 100:.2f}%"
    )

    print(
        f"CNN Precision                   : "
        f"{cnn_precision * 100:.2f}%"
    )

    print(
        f"CNN Recall                      : "
        f"{cnn_recall * 100:.2f}%"
    )

    print(
        f"CNN F1 Score                    : "
        f"{cnn_f1 * 100:.2f}%"
    )

else:

    cnn_accuracy = np.nan
    cnn_precision = np.nan
    cnn_recall = np.nan
    cnn_f1 = np.nan

    print("No clean images were passed to ResNet50.")


# ============================================================
# 19. CNN CLASSIFICATION REPORT
# ============================================================

if len(cnn_true) > 0:

    print("\n===================================================")
    print("CNN CLASSIFICATION REPORT")
    print("===================================================")

    print(
        classification_report(
            cnn_true,
            cnn_pred,
            labels=np.arange(
                len(label_encoder.classes_)
            ),
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )


# ============================================================
# 20. SAVE IMAGE LEVEL RESULTS
# ============================================================

RESULTS_PATH = (
    "/kaggle/working/"
    "PRE_detector_test_results.csv"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("\nSaved image-level results:")
print(RESULTS_PATH)


# ============================================================
# 21. SAVE KNN SUMMARY
# ============================================================

knn_summary = pd.DataFrame({

    "Metric": [

        "Total Images",
        "Clean Images",
        "Adversarial Images",
        "Adversarial Detected",
        "Clean Passed",
        "TP",
        "TN",
        "FP",
        "FN",
        "Accuracy",
        "Precision",
        "Recall / TPR",
        "FNR",
        "FPR",
        "F1 Score",
        "AUC"

    ],

    "Value": [

        total_images,
        clean_images,
        adversarial_images,
        adversarial_detected,
        clean_passed,
        TP,
        TN,
        FP,
        FN,
        accuracy,
        precision,
        recall,
        fnr,
        fpr,
        f1,
        auc_value

    ]

})


KNN_SUMMARY_PATH = (
    "/kaggle/working/"
    "PRE_detection_summary.csv"
)

knn_summary.to_csv(
    KNN_SUMMARY_PATH,
    index=False
)

print("Saved KNN summary:")
print(KNN_SUMMARY_PATH)


# ============================================================
# 22. SAVE CNN SUMMARY
# ============================================================

cnn_summary = pd.DataFrame({

    "Metric": [

        "Clean-passed images sent to CNN",
        "CNN Accuracy",
        "CNN Precision",
        "CNN Recall",
        "CNN F1 Score"

    ],

    "Value": [

        len(cnn_true),
        cnn_accuracy,
        cnn_precision,
        cnn_recall,
        cnn_f1

    ]

})


CNN_SUMMARY_PATH = (
    "/kaggle/working/"
    "PRE_CNN_summary.csv"
)

cnn_summary.to_csv(
    CNN_SUMMARY_PATH,
    index=False
)

print("Saved CNN summary:")
print(CNN_SUMMARY_PATH)


# ============================================================
# 23. SAVE ROC DATA
# ============================================================

if len(fpr_curve) > 0:

    roc_df = pd.DataFrame({

        "False Positive Rate": fpr_curve,

        "True Positive Rate": tpr_curve,

        "Threshold": thresholds

    })

else:

    roc_df = pd.DataFrame({

        "False Positive Rate": [],

        "True Positive Rate": [],

        "Threshold": []

    })


ROC_DATA_PATH = (
    "/kaggle/working/"
    "PRE_ROC_data.csv"
)

roc_df.to_csv(
    ROC_DATA_PATH,
    index=False
)

print("Saved ROC data:")
print(ROC_DATA_PATH)


# ============================================================
# 24. PLOT PRE ROC CURVE
# ============================================================

if len(fpr_curve) > 0:

    plt.figure(figsize=(7, 6))

    plt.plot(
        fpr_curve,
        tpr_curve,
        linewidth=2,
        label=f"Pre-KNN (AUC = {auc_value:.4f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1
    )

    plt.xlabel("False Positive Rate")

    plt.ylabel("True Positive Rate")

    plt.title(
        "ROC Curve - Pre-Model Adversarial Detection"
    )

    plt.legend(
        loc="lower right"
    )

    plt.grid(True)

    plt.tight_layout()

    ROC_IMAGE_PATH = (
        "/kaggle/working/"
        "PRE_ROC_curve.png"
    )

    plt.savefig(
        ROC_IMAGE_PATH,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("\nROC curve saved:")
    print(ROC_IMAGE_PATH)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n===================================================")
print("PRE-MODEL PIPELINE COMPLETED")
print("===================================================")

print("\nKNN:")
print(" - Separate Clean/Adversarial dataset used for training")
print(" - Test dataset NOT used for KNN training")
print(" - StandardScaler fitted only on KNN training data")
print(" - KNN K =", KNN_K)
print(" - Distance metric = Euclidean")
print(" - Weighting = Distance")

print("\nTesting:")
print(" - Mixed dataset used only for evaluation")
print(" - First 30 images/class = Clean")
print(" - Next 10 images/class = Adversarial")

print("\nCNN:")
print(" - Only images classified as Clean by KNN")
print("   are passed to ResNet50.")

print("\nFiles generated:")
print("1.", RESULTS_PATH)
print("2.", KNN_SUMMARY_PATH)
print("3.", CNN_SUMMARY_PATH)
print("4.", ROC_DATA_PATH)

if len(fpr_curve) > 0:
    print("5.", ROC_IMAGE_PATH)

print("\n===================================================")

# *#
# POST-MODEL ADVERSARIAL EXAMPLE DETECTION using all crop epsilon***

In [ ]:
# ============================================================
# POST-MODEL ADVERSARIAL EXAMPLE DETECTION
# ============================================================
# Separate KNN training dataset:
#       Clean / Adversarial
#
# KNN features:
#       ResNet50 -> GlobalAveragePooling2D -> 2048-D
#
# Evaluation dataset:
#       Original disease-class folders
#       First 30 images = Clean
#       Next 10 images  = Adversarial
#
# KNN-clean images are passed to ResNet50
# KNN-adversarial images are detected and removed
# ============================================================


import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import tensorflow as tf
import joblib
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)


# ============================================================
# 1. PATHS
# ============================================================

# ------------------------------------------------------------
# SEPARATE DATASET ONLY FOR KNN TRAINING
#
# Structure:
#
# KNN_TRAIN_PATH/
# ├── Clean/
# │    ├── image1.jpg
# │    ├── image2.jpg
# │    └── ...
# │
# └── Adversarial/
#      ├── image1.jpg
#      ├── image2.jpg
#      └── ...
# ------------------------------------------------------------

KNN_TRAIN_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/combine-tomato"
)


# ------------------------------------------------------------
# MIXED DATASET ONLY FOR FINAL EVALUATION
#
# Structure:
#
# TEST_PATH/
# ├── Disease_Class_1/
# │    ├── first 30 = Clean
# │    └── next 10  = Adversarial
# │
# ├── Disease_Class_2/
# │    ├── first 30 = Clean
# │    └── next 10  = Adversarial
# │
# └── ...
# ------------------------------------------------------------

TEST_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Tomato_test"
)


# ------------------------------------------------------------
# TRAINED RESNET50
# ------------------------------------------------------------

RESNET_PATH = (
    "/kaggle/working/"
    "resnet50_6class_20epochs.keras"
)


# ============================================================
# 2. PARAMETERS
# ============================================================

IMAGE_SIZE = (224, 224)

K = 5

CLEAN_PER_CLASS = 30

ADV_PER_CLASS = 10


VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# 3. LOAD TRAINED RESNET50
# ============================================================

print("=" * 70)
print("LOADING RESNET50")
print("=" * 70)


resnet = tf.keras.models.load_model(
    RESNET_PATH
)


print(
    "ResNet50 loaded successfully."
)


print("\nResNet input shape:")
print(
    resnet.input_shape
)


print("\nResNet output shape:")
print(
    resnet.output_shape
)


# ============================================================
# 4. FIND GLOBAL AVERAGE POOLING LAYER
# ============================================================

print("\n")
print("=" * 70)
print("CREATING FEATURE EXTRACTOR")
print("=" * 70)


# Same method as your original POST code.
# Find the actual GlobalAveragePooling2D layer.

feature_layer = None


for layer in reversed(
    resnet.layers
):

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):

        feature_layer = layer

        break


if feature_layer is None:

    raise ValueError(
        "GlobalAveragePooling2D layer not found."
    )


feature_extractor = Model(
    inputs=resnet.input,
    outputs=feature_layer.output
)


print(
    "Feature extraction layer:",
    feature_layer.name
)


print(
    "Feature output shape:",
    feature_extractor.output_shape
)


# ============================================================
# 5. IMAGE LIST FUNCTION
# ============================================================

def get_images(folder):

    folder = Path(folder)

    if not folder.exists():

        return []


    return sorted(
        [
            file

            for file in folder.iterdir()

            if file.is_file()

            and file.suffix.lower()
            in VALID_EXTENSIONS
        ]
    )


# ============================================================
# 6. RESNET FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_single_feature(
    image_path
):

    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")


    # --------------------------------------------------------
    # Resize
    # --------------------------------------------------------

    image = image.resize(
        IMAGE_SIZE
    )


    # --------------------------------------------------------
    # Convert to NumPy
    # --------------------------------------------------------

    image_array = np.asarray(
        image,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Same /255 preprocessing
    # --------------------------------------------------------

    image_array = (
        image_array / 255.0
    )


    # --------------------------------------------------------
    # Add batch dimension
    # --------------------------------------------------------

    batch = np.expand_dims(
        image_array,
        axis=0
    )


    # --------------------------------------------------------
    # Extract 2048-D GAP feature
    # --------------------------------------------------------

    feature = (
        feature_extractor.predict(
            batch,
            verbose=0
        )
    )


    # --------------------------------------------------------
    # Flatten
    # --------------------------------------------------------

    feature = feature.flatten()


    return feature, batch


# ============================================================
# 7. LOAD SEPARATE KNN TRAINING DATASET
# ============================================================

print("\n")
print("=" * 70)
print("LOADING SEPARATE KNN TRAINING DATASET")
print("=" * 70)


CLEAN_FOLDER = (
    KNN_TRAIN_PATH / "Clean"
)

ADV_FOLDER = (
    KNN_TRAIN_PATH / "Adversarial"
)


# ============================================================
# 8. EXTRACT CLEAN FEATURES
# ============================================================

clean_images = get_images(
    CLEAN_FOLDER
)


print("\n")
print("-" * 70)
print("KNN TRAINING - CLEAN")
print("-" * 70)

print(
    "Clean images:",
    len(clean_images)
)


X_clean = []

y_clean = []


for i, image_path in enumerate(
    clean_images
):

    try:

        feature, _ = (
            extract_single_feature(
                image_path
            )
        )


        X_clean.append(
            feature
        )

        # Clean = 0
        y_clean.append(
            0
        )


        if (
            (i + 1) % 50 == 0
            or
            i == len(clean_images) - 1
        ):

            print(
                f"Processed "
                f"{i + 1}/"
                f"{len(clean_images)}"
            )


    except Exception as e:

        print(
            f"Error processing "
            f"{image_path}: {e}"
        )


X_clean = np.array(
    X_clean,
    dtype=np.float32
)


y_clean = np.array(
    y_clean,
    dtype=np.int32
)


# ============================================================
# 9. EXTRACT ADVERSARIAL FEATURES
# ============================================================

adv_images = get_images(
    ADV_FOLDER
)


print("\n")
print("-" * 70)
print("KNN TRAINING - ADVERSARIAL")
print("-" * 70)

print(
    "Adversarial images:",
    len(adv_images)
)


X_adv = []

y_adv = []


for i, image_path in enumerate(
    adv_images
):

    try:

        feature, _ = (
            extract_single_feature(
                image_path
            )
        )


        X_adv.append(
            feature
        )

        # Adversarial = 1
        y_adv.append(
            1
        )


        if (
            (i + 1) % 50 == 0
            or
            i == len(adv_images) - 1
        ):

            print(
                f"Processed "
                f"{i + 1}/"
                f"{len(adv_images)}"
            )


    except Exception as e:

        print(
            f"Error processing "
            f"{image_path}: {e}"
        )


X_adv = np.array(
    X_adv,
    dtype=np.float32
)


y_adv = np.array(
    y_adv,
    dtype=np.int32
)


# ============================================================
# 10. COMBINE KNN TRAINING FEATURES
# ============================================================

X_train = np.concatenate(
    [
        X_clean,
        X_adv
    ],
    axis=0
)


y_train = np.concatenate(
    [
        y_clean,
        y_adv
    ],
    axis=0
)


print("\n")
print("=" * 70)
print("POST KNN TRAINING DATASET")
print("=" * 70)


print(
    "Clean images       :",
    len(X_clean)
)


print(
    "Adversarial images :",
    len(X_adv)
)


print(
    "Total images       :",
    len(X_train)
)


print(
    "Feature dimension  :",
    X_train.shape[1]
)


print(
    "Expected dimension : 2048"
)


# ============================================================
# 11. STANDARD SCALER
# ============================================================

print("\n")
print("=" * 70)
print("FITTING POST STANDARD SCALER")
print("=" * 70)


scaler_post = StandardScaler()


# IMPORTANT:
# FIT ONLY ON KNN TRAINING DATA

X_train_scaled = (
    scaler_post.fit_transform(
        X_train
    )
)


print(
    "Scaled feature shape:",
    X_train_scaled.shape
)


# ============================================================
# 12. TRAIN POST KNN
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING POST KNN")
print("=" * 70)


knn_post = KNeighborsClassifier(
    n_neighbors=K,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)


knn_post.fit(
    X_train_scaled,
    y_train
)


print(
    "POST KNN training completed."
)


print(
    "K =",
    K
)


print(
    "Metric = Euclidean"
)


print(
    "Weights = Distance"
)


# ============================================================
# 13. SAVE KNN + SCALER
# ============================================================

knn_path = (
    "/kaggle/working/"
    "post_knn_resnet_features.pkl"
)


scaler_path = (
    "/kaggle/working/"
    "post_knn_scaler.pkl"
)


joblib.dump(
    knn_post,
    knn_path
)


joblib.dump(
    scaler_post,
    scaler_path
)


print("\n")
print("=" * 70)
print("POST KNN MODEL SAVED")
print("=" * 70)


print(
    "KNN:",
    knn_path
)


print(
    "Scaler:",
    scaler_path
)


# ============================================================
# 14. GET TEST DATASET CLASSES
# ============================================================

class_folders = sorted(
    [
        folder

        for folder in TEST_PATH.iterdir()

        if folder.is_dir()
    ],
    key=lambda x: x.name
)


print("\n")
print("=" * 70)
print("TEST DATASET CLASSES")
print("=" * 70)


for folder in class_folders:

    print(
        folder.name
    )


# ============================================================
# 15. CHECK LABEL ENCODER
# ============================================================

if "label_encoder" not in globals():

    raise NameError(
        "label_encoder is not available. "
        "Use the same LabelEncoder that was used "
        "during ResNet50 training."
    )


print("\n")
print("=" * 70)
print("LABEL ENCODER MAPPING")
print("=" * 70)


for index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{class_name} --> {index}"
    )


# ============================================================
# 16. RESULT STORAGE
# ============================================================

total_images = 0

clean_ground_truth = 0

adversarial_ground_truth = 0

adversarial_detected = 0

clean_passed = 0


# Detection labels
y_true_detection = []

y_pred_detection = []

y_score_detection = []


# CNN labels
cnn_true_labels = []

cnn_pred_labels = []


# Image-level results
results = []


# ============================================================
# 17. START TESTING
# ============================================================

print("\n")
print("=" * 70)
print("STARTING POST DETECTOR TESTING")
print("=" * 70)


for class_folder in class_folders:


    # ========================================================
    # DISEASE CLASS
    # ========================================================

    true_class_name = (
        class_folder.name
    )


    # --------------------------------------------------------
    # Same LabelEncoder used during CNN training
    # --------------------------------------------------------

    try:

        true_class_index = int(
            label_encoder.transform(
                [true_class_name]
            )[0]
        )

    except ValueError:

        raise ValueError(
            f"Class '{true_class_name}' "
            f"was not present in the "
            f"LabelEncoder used during CNN training."
        )


    # ========================================================
    # GET IMAGES
    # ========================================================

    image_files = get_images(
        class_folder
    )


    print("\n")
    print("-" * 70)

    print(
        "Class:",
        true_class_name
    )

    print(
        "Encoded label:",
        true_class_index
    )

    print(
        "Images:",
        len(image_files)
    )

    print("-" * 70)


    # --------------------------------------------------------
    # Check 40 images
    # --------------------------------------------------------

    if len(image_files) != 40:

        print(
            "WARNING:",
            true_class_name,
            "contains",
            len(image_files),
            "images instead of 40."
        )


    # ========================================================
    # PROCESS EACH IMAGE
    # ========================================================

    for index, image_path in enumerate(
        image_files
    ):


        total_images += 1


        # ====================================================
        # GROUND TRUTH
        # ====================================================
        #
        # First 30 = Clean
        # Next 10  = Adversarial
        # ====================================================

        if index < CLEAN_PER_CLASS:

            true_detection_label = 0

            true_detection_name = (
                "Clean"
            )

            clean_ground_truth += 1

        else:

            true_detection_label = 1

            true_detection_name = (
                "Adversarial"
            )

            adversarial_ground_truth += 1


        try:

            # =================================================
            # RESNET PREPROCESSING + FEATURE EXTRACTION
            # =================================================

            feature, batch = (
                extract_single_feature(
                    image_path
                )
            )


            # -------------------------------------------------
            # Reshape
            # -------------------------------------------------

            feature = feature.reshape(
                1,
                -1
            )


            # =================================================
            # SAME SCALER TRAINED ON KNN TRAINING DATA
            # =================================================

            feature_scaled = (
                scaler_post.transform(
                    feature
                )
            )


            # =================================================
            # POST KNN PREDICTION
            # =================================================

            knn_prediction = int(
                knn_post.predict(
                    feature_scaled
                )[0]
            )


            # =================================================
            # ADVERSARIAL SCORE
            # =================================================

            probabilities = (
                knn_post.predict_proba(
                    feature_scaled
                )[0]
            )


            class_indices = list(
                knn_post.classes_
            )


            if 1 in class_indices:

                adv_index = (
                    class_indices.index(1)
                )

                adversarial_score = float(
                    probabilities[
                        adv_index
                    ]
                )

            else:

                adversarial_score = 0.0


            # =================================================
            # STORE DETECTION LABELS
            # =================================================

            y_true_detection.append(
                true_detection_label
            )

            y_pred_detection.append(
                knn_prediction
            )

            y_score_detection.append(
                adversarial_score
            )


            # =================================================
            # KNN = ADVERSARIAL
            # =================================================

            if knn_prediction == 1:

                adversarial_detected += 1


                results.append({

                    "image":
                        str(image_path),

                    "image_index":
                        index + 1,

                    "true_class":
                        true_class_name,

                    "true_class_index":
                        true_class_index,

                    "ground_truth_detection":
                        true_detection_name,

                    "ground_truth_detection_label":
                        true_detection_label,

                    "KNN_prediction":
                        "Adversarial",

                    "KNN_prediction_label":
                        1,

                    "adversarial_score":
                        adversarial_score,

                    "CNN_prediction":
                        "Not Tested"

                })


                # IMPORTANT:
                # Adversarial image is NOT
                # sent to CNN.

                continue


            # =================================================
            # KNN = CLEAN
            # =================================================

            clean_passed += 1


            # =================================================
            # SEND CLEAN IMAGE TO RESNET50
            # =================================================

            cnn_output = (
                resnet.predict(
                    batch,
                    verbose=0
                )
            )


            # =================================================
            # CNN PREDICTION
            # =================================================

            predicted_class_index = int(
                np.argmax(
                    cnn_output,
                    axis=1
                )[0]
            )


            # -------------------------------------------------
            # Store CNN labels
            # -------------------------------------------------

            cnn_true_labels.append(
                true_class_index
            )

            cnn_pred_labels.append(
                predicted_class_index
            )


            # -------------------------------------------------
            # Convert prediction to class name
            # -------------------------------------------------

            predicted_class_name = (
                label_encoder.inverse_transform(
                    [
                        predicted_class_index
                    ]
                )[0]
            )


            # =================================================
            # SAVE IMAGE RESULT
            # =================================================

            results.append({

                "image":
                    str(image_path),

                "image_index":
                    index + 1,

                "true_class":
                    true_class_name,

                "true_class_index":
                    true_class_index,

                "ground_truth_detection":
                    true_detection_name,

                "ground_truth_detection_label":
                    true_detection_label,

                "KNN_prediction":
                    "Clean",

                "KNN_prediction_label":
                    0,

                "adversarial_score":
                    adversarial_score,

                "CNN_prediction":
                    predicted_class_name,

                "CNN_prediction_index":
                    predicted_class_index

            })


        except Exception as e:

            print(
                f"Error processing "
                f"{image_path}: {e}"
            )


# ============================================================
# 18. CONVERT ARRAYS
# ============================================================

y_true_detection = np.array(
    y_true_detection
)


y_pred_detection = np.array(
    y_pred_detection
)


y_score_detection = np.array(
    y_score_detection
)


cnn_true_labels = np.array(
    cnn_true_labels
)


cnn_pred_labels = np.array(
    cnn_pred_labels
)


# ============================================================
# 19. POST-KNN CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true_detection,
    y_pred_detection,
    labels=[0, 1]
)


TN, FP, FN, TP = (
    cm.ravel()
)


print("\n")
print("=" * 70)
print("POST-KNN DETECTION RESULTS")
print("=" * 70)


print("\nConfusion Matrix:")

print(
    cm
)


print("\nTN:", TN)

print("FP:", FP)

print("FN:", FN)

print("TP:", TP)


# ============================================================
# 20. KNN METRICS
# ============================================================

accuracy = accuracy_score(
    y_true_detection,
    y_pred_detection
)


precision = precision_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)


recall = recall_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)


f1 = f1_score(
    y_true_detection,
    y_pred_detection,
    zero_division=0
)


# TPR
if (TP + FN) > 0:

    tpr = (
        TP /
        (TP + FN)
    )

else:

    tpr = 0.0


# FNR
if (TP + FN) > 0:

    fnr = (
        FN /
        (TP + FN)
    )

else:

    fnr = 0.0


# FPR
if (FP + TN) > 0:

    fpr = (
        FP /
        (FP + TN)
    )

else:

    fpr = 0.0


# ============================================================
# 21. ROC + AUC
# ============================================================

if len(
    np.unique(
        y_true_detection
    )
) == 2:


    fpr_curve, tpr_curve, thresholds = (
        roc_curve(
            y_true_detection,
            y_score_detection
        )
    )


    auc_value = roc_auc_score(
        y_true_detection,
        y_score_detection
    )


else:

    fpr_curve = np.array([])

    tpr_curve = np.array([])

    thresholds = np.array([])

    auc_value = np.nan


# ============================================================
# 22. PRINT KNN RESULTS
# ============================================================

print(
    f"\nTotal Images          : "
    f"{total_images}"
)


print(
    f"Clean Images          : "
    f"{clean_ground_truth}"
)


print(
    f"Adversarial Images    : "
    f"{adversarial_ground_truth}"
)


print(
    f"Adversarial Detected  : "
    f"{adversarial_detected}"
)


print(
    f"Clean Passed          : "
    f"{clean_passed}"
)


print(
    f"TP                    : "
    f"{TP}"
)


print(
    f"TN                    : "
    f"{TN}"
)


print(
    f"FP                    : "
    f"{FP}"
)


print(
    f"FN                    : "
    f"{FN}"
)


print(
    f"Accuracy              : "
    f"{accuracy * 100:.2f}%"
)


print(
    f"Precision             : "
    f"{precision * 100:.2f}%"
)


print(
    f"Recall / TPR          : "
    f"{recall * 100:.2f}%"
)


print(
    f"FNR                   : "
    f"{fnr * 100:.2f}%"
)


print(
    f"FPR                   : "
    f"{fpr * 100:.2f}%"
)


print(
    f"F1 Score              : "
    f"{f1 * 100:.2f}%"
)


print(
    f"AUC                   : "
    f"{auc_value:.4f}"
)


# ============================================================
# 23. CNN METRICS
# ============================================================

print("\n")
print("=" * 70)
print("RESNET50 AFTER POST DETECTION")
print("=" * 70)


if len(cnn_true_labels) > 0:


    cnn_accuracy = accuracy_score(
        cnn_true_labels,
        cnn_pred_labels
    )


    cnn_precision = precision_score(
        cnn_true_labels,
        cnn_pred_labels,
        average="weighted",
        zero_division=0
    )


    cnn_recall = recall_score(
        cnn_true_labels,
        cnn_pred_labels,
        average="weighted",
        zero_division=0
    )


    cnn_f1 = f1_score(
        cnn_true_labels,
        cnn_pred_labels,
        average="weighted",
        zero_division=0
    )


    print(
        "Images passed to CNN :",
        len(cnn_true_labels)
    )


    print(
        f"CNN Accuracy         : "
        f"{cnn_accuracy * 100:.2f}%"
    )


    print(
        f"CNN Precision        : "
        f"{cnn_precision * 100:.2f}%"
    )


    print(
        f"CNN Recall           : "
        f"{cnn_recall * 100:.2f}%"
    )


    print(
        f"CNN F1 Score         : "
        f"{cnn_f1 * 100:.2f}%"
    )


else:

    cnn_accuracy = np.nan

    cnn_precision = np.nan

    cnn_recall = np.nan

    cnn_f1 = np.nan


    print(
        "No Clean images were "
        "passed to CNN."
    )


# ============================================================
# 24. CNN CLASSIFICATION REPORT
# ============================================================

if len(cnn_true_labels) > 0:


    print("\n")
    print("=" * 70)
    print("CNN CLASSIFICATION REPORT")
    print("=" * 70)


    print(
        classification_report(
            cnn_true_labels,
            cnn_pred_labels,
            labels=list(
                range(
                    len(
                        label_encoder.classes_
                    )
                )
            ),
            target_names=(
                label_encoder.classes_
            ),
            digits=4,
            zero_division=0
        )
    )


# ============================================================
# 25. CNN CONFUSION MATRIX
# ============================================================

if len(cnn_true_labels) > 0:


    cnn_cm = confusion_matrix(
        cnn_true_labels,
        cnn_pred_labels,
        labels=list(
            range(
                len(
                    label_encoder.classes_
                )
            )
        )
    )


    print("\n")
    print("=" * 70)
    print("CNN CONFUSION MATRIX")
    print("=" * 70)


    print(
        cnn_cm
    )


# ============================================================
# 26. SAVE IMAGE-LEVEL RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)


results_path = (
    "/kaggle/working/"
    "POST_detector_test_results.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


print("\n")
print(
    "Image-level results saved:"
)

print(
    results_path
)


# ============================================================
# 27. SAVE KNN SUMMARY
# ============================================================

knn_summary = pd.DataFrame({

    "Metric": [

        "Total Images",

        "Clean Images",

        "Adversarial Images",

        "Adversarial Detected",

        "Clean Passed",

        "TP",

        "TN",

        "FP",

        "FN",

        "Accuracy",

        "Precision",

        "Recall / TPR",

        "FNR",

        "FPR",

        "F1 Score",

        "AUC"

    ],

    "Value": [

        total_images,

        clean_ground_truth,

        adversarial_ground_truth,

        adversarial_detected,

        clean_passed,

        TP,

        TN,

        FP,

        FN,

        accuracy,

        precision,

        recall,

        fnr,

        fpr,

        f1,

        auc_value

    ]

})


knn_summary_path = (
    "/kaggle/working/"
    "POST_detection_summary.csv"
)


knn_summary.to_csv(
    knn_summary_path,
    index=False
)


print(
    "KNN summary saved:"
)

print(
    knn_summary_path
)


# ============================================================
# 28. SAVE CNN SUMMARY
# ============================================================

cnn_summary = pd.DataFrame({

    "Metric": [

        "Images Passed to CNN",

        "CNN Accuracy",

        "CNN Precision",

        "CNN Recall",

        "CNN F1 Score"

    ],

    "Value": [

        len(cnn_true_labels),

        cnn_accuracy,

        cnn_precision,

        cnn_recall,

        cnn_f1

    ]

})


cnn_summary_path = (
    "/kaggle/working/"
    "POST_CNN_summary.csv"
)


cnn_summary.to_csv(
    cnn_summary_path,
    index=False
)


print(
    "CNN summary saved:"
)

print(
    cnn_summary_path
)


# ============================================================
# 29. SAVE ROC DATA
# ============================================================

roc_df = pd.DataFrame({

    "False Positive Rate":
        fpr_curve,

    "True Positive Rate":
        tpr_curve,

    "Threshold":
        thresholds

})


roc_data_path = (
    "/kaggle/working/"
    "POST_ROC_data.csv"
)


roc_df.to_csv(
    roc_data_path,
    index=False
)


print(
    "ROC data saved:"
)

print(
    roc_data_path
)


# ============================================================
# 30. PLOT ROC CURVE
# ============================================================

if len(fpr_curve) > 0:


    plt.figure(
        figsize=(7, 6)
    )


    plt.plot(
        fpr_curve,
        tpr_curve,
        linewidth=2,
        label=(
            f"Post-KNN "
            f"(AUC = {auc_value:.4f})"
        )
    )


    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1
    )


    plt.xlabel(
        "False Positive Rate"
    )


    plt.ylabel(
        "True Positive Rate"
    )


    plt.title(
        "ROC Curve - "
        "Post-Model Adversarial Detection"
    )


    plt.legend(
        loc="lower right"
    )


    plt.grid(
        True
    )


    plt.tight_layout()


    roc_image_path = (
        "/kaggle/working/"
        "POST_ROC_curve.png"
    )


    plt.savefig(
        roc_image_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


    print(
        "\nROC curve saved:"
    )

    print(
        roc_image_path
    )


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("POST-MODEL PIPELINE COMPLETED")
print("=" * 70)


print("\nKNN TRAINING DATASET")
print(
    "Clean + Adversarial only"
)


print("\nFEATURE EXTRACTION")
print(
    "ResNet50 -> GlobalAveragePooling2D -> 2048-D"
)


print("\nKNN")
print(
    "K =",
    K
)

print(
    "Distance = Euclidean"
)

print(
    "Weights = Distance"
)


print("\nTEST DATASET")
print(
    "First 30 images/class = Clean"
)

print(
    "Next 10 images/class = Adversarial"
)


print("\nROUTING")
print(
    "KNN = Adversarial -> Detected"
)

print(
    "KNN = Clean -> ResNet50"
)


print("\nFILES GENERATED")

print(
    "1.",
    results_path
)

print(
    "2.",
    knn_summary_path
)

print(
    "3.",
    cnn_summary_path
)

print(
    "4.",
    roc_data_path
)

if len(fpr_curve) > 0:

    print(
        "5.",
        roc_image_path
    )

print(
    "6.",
    knn_path
)

print(
    "7.",
    scaler_path
)


print("\n")
print("=" * 70)

# *testing on mixed data from skreach***

# *pre***

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier


# ============================================================
# 1. DATASET PATH
# ============================================================

DATASET_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/combine-apple"
)

CLEAN_FOLDER = DATASET_PATH / "Clean"
ADVERSARIAL_FOLDER = DATASET_PATH / "Adversarial"

IMAGE_SIZE = (224, 224)

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# 2. FUNCTION TO LOAD AND PREPROCESS IMAGES
# ============================================================

def load_images(folder, label):

    X = []
    y = []

    image_files = sorted([
        file
        for file in folder.iterdir()
        if file.is_file()
        and file.suffix.lower() in VALID_EXTENSIONS
    ])

    print(f"{folder.name}: {len(image_files)} images")

    for image_path in image_files:

        try:

            # Load image
            image = Image.open(
                image_path
            ).convert("RGB")

            # Resize to 224 × 224
            image = image.resize(
                IMAGE_SIZE
            )

            # Convert to NumPy array
            image_array = np.asarray(
                image,
                dtype=np.float32
            )

            # Normalize pixel values: 0–255 → 0–1
            image_array = image_array / 255.0

            # Flatten:
            # 224 × 224 × 3 → 150528
            image_vector = image_array.flatten()

            X.append(image_vector)
            y.append(label)

        except Exception as e:

            print(
                f"Error loading {image_path}: {e}"
            )


    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.int32)
    )


# ============================================================
# 3. LOAD CLEAN IMAGES
#    Label = 0
# ============================================================

X_clean, y_clean = load_images(
    CLEAN_FOLDER,
    label=0
)


# ============================================================
# 4. LOAD ADVERSARIAL IMAGES
#    Label = 1
# ============================================================

X_adversarial, y_adversarial = load_images(
    ADVERSARIAL_FOLDER,
    label=1
)


# ============================================================
# 5. COMBINE BOTH CLASSES
# ============================================================

X_train = np.concatenate(
    [
        X_clean,
        X_adversarial
    ],
    axis=0
)

y_train = np.concatenate(
    [
        y_clean,
        y_adversarial
    ],
    axis=0
)


# ============================================================
# 6. CHECK TRAINING DATA
# ============================================================

print("\n" + "=" * 70)
print("KNN TRAINING DATA")
print("=" * 70)

print("Clean images       :", len(X_clean))
print("Adversarial images :", len(X_adversarial))
print("Total images       :", len(X_train))

print("\nX_train shape      :", X_train.shape)
print("y_train shape      :", y_train.shape)

print("\nExpected feature size: 150528")


# ============================================================
# 7. TRAIN KNN
# ============================================================

knn_post = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

knn_post.fit(
    X_train,
    y_train
)


# ============================================================
# 8. TRAINING COMPLETED
# ============================================================

print("\n" + "=" * 70)
print("KNN TRAINING COMPLETED")
print("=" * 70)

print("KNN classes:", knn_post.classes_)
print("Number of training samples:", len(X_train))
print("Number of features:", X_train.shape[1])

In [ ]:

import os
import cv2
import numpy as np

train_folder = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Apple/Train'
test_folder = '/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Apple_test'
#test_folder1 = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Tomato/Test'

IMG_SIZE = (224, 224)
MAX_PER_CLASS = 10000   # 🔴 IMPORTANT

def load_images_limited(folder, max_per_class=None):
    images = []
    labels = []
    
    for label in sorted(os.listdir(folder)):
        label_path = os.path.join(folder, label)
        if not os.path.isdir(label_path):
            continue
        
        count = 0
        for filename in os.listdir(label_path):
            if max_per_class is not None and count >= max_per_class:
                break
            
            img_path = os.path.join(label_path, filename)
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(label)
            count += 1

        print(f"Loaded {count} images from class '{label}'")
    
    return np.array(images), np.array(labels)

# 🔹 TRAIN: only 500 images per class
x_train, y_train = load_images_limited(train_folder, MAX_PER_CLASS)

# 🔹 TEST (no limit)
x_test, y_test = load_images_limited(test_folder)
#x_test_1, y_test_1 = load_images_limited(test_folder1)

# 🔹 Verify shapes
print("\nFinal Shapes:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)
#print("x_test_1:", x_test_1.shape)
#print("y_test_1:", y_test_1.shape)



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels in y_train
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform the labels in y_test
y_test_encoded = label_encoder.transform(y_test)
#y_test_encoded_1 = label_encoder.transform(y_test_1)





In [ ]:
X_test=x_test
y_test=y_test_encoded

In [ ]:
import numpy as np


# ============================================================
# 1. ORIGINAL TEST DATA
# ============================================================

print("=" * 70)
print("ORIGINAL TEST DATA")
print("=" * 70)

print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)


# ============================================================
# 2. CREATE TEMPORARY 2-D COPY FOR KNN
# ============================================================

# Original X_test is NOT changed.
# Only a temporary 2-D copy is created for KNN.

X_test_2D = X_test.reshape(
    len(X_test),
    -1
)

print("\nTemporary X_test for KNN:")
print("X_test_2D shape :", X_test_2D.shape)


# ============================================================
# 3. SAME PREPROCESSING USED DURING KNN TRAINING
# ============================================================

# Training images were converted to float32
# and normalized from 0-255 to 0-1.

X_test_2D = X_test_2D.astype(
    np.float32
)

X_test_2D = X_test_2D / 255.0


# ============================================================
# 4. KNN PREDICTION
# ============================================================

knn_prediction = knn_post.predict(
    X_test_2D
)


# ============================================================
# 5. FIND ADVERSARIAL INDICES
# ============================================================

# KNN:
# 0 = Clean
# 1 = Adversarial

adversarial_indices = np.where(
    knn_prediction == 1
)[0]


# ============================================================
# 6. DISPLAY INDICES
# ============================================================

print("\n" + "=" * 70)
print("KNN FILTERING")
print("=" * 70)

print(
    "Total test images          :",
    len(X_test)
)

print(
    "Adversarial indices found :",
    len(adversarial_indices)
)

print(
    "Clean images remaining    :",
    len(X_test) - len(adversarial_indices)
)

print("\nAdversarial indices:")
print(adversarial_indices)


# ============================================================
# 7. DELETE ADVERSARIAL IMAGES
#    FROM ORIGINAL X_TEST
# ============================================================

X_test_clean = np.delete(
    X_test,
    adversarial_indices,
    axis=0
)


# ============================================================
# 8. DELETE CORRESPONDING LABELS
#    FROM ORIGINAL Y_TEST
# ============================================================

y_test_clean = np.delete(
    y_test,
    adversarial_indices,
    axis=0
)


# ============================================================
# 9. VERIFY
# ============================================================

print("\n" + "=" * 70)
print("FINAL CLEAN TEST DATA")
print("=" * 70)

print(
    "Original X_test shape :",
    X_test.shape
)

print(
    "Original y_test shape :",
    y_test.shape
)

print(
    "Clean X_test shape    :",
    X_test_clean.shape
)

print(
    "Clean y_test shape    :",
    y_test_clean.shape
)


# ============================================================
# 10. ALIGNMENT CHECK
# ============================================================

if len(X_test_clean) != len(y_test_clean):

    raise ValueError(
        "X_test_clean and y_test_clean are not aligned!"
    )

print("\nX_test_clean and y_test_clean are perfectly aligned.")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize


# ============================================================
# PATHS
# ============================================================

model_path = "/kaggle/working/resnet50_6class_20epochs.keras"

model = load_model(model_path)

print("Model loaded successfully!")

# Check data
print("Clean X_test shape :", X_test_clean.shape)
print("Clean y_test shape :", y_test_clean.shape)

# ResNet prediction
y_prob = model.predict(
    X_test_clean,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)

print("Prediction shape :", y_pred.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_true = le.fit_transform(y_test_clean)

class_names = le.classes_

print("Classes:", class_names)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 50)
print("RESNET RESULTS ON CLEAN DATA")
print("=" * 50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

In [ ]:
num_classes = len(class_names)

y_true_onehot = label_binarize(
    y_true,
    classes=np.arange(num_classes)
)

auc_score = roc_auc_score(
    y_true_onehot,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

print(f"AUC       : {auc_score:.4f}")

In [ ]:
class_names = label_encoder.classes_
class_names = label_encoder.classes_
num_classes = len(class_names)

print(class_names)

In [ ]:
plt.figure(figsize=(9, 7))

for i in range(num_classes):

    fpr, tpr, _ = roc_curve(
        y_true_onehot[:, i],
        y_prob[:, i]
    )

    auc_i = roc_auc_score(
        y_true_onehot[:, i],
        y_prob[:, i]
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{class_names[i]} (AUC = {auc_i:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - ResNet on Clean Test Data")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred)


plt.figure(figsize=(8, 6))

plt.imshow(
    cm,
    cmap="Blues"
)


plt.title("Confusion Matrix - ResNet")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    range(num_classes),
    class_names,
    rotation=90
)

plt.yticks(
    range(num_classes),
    class_names
)

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()

# *standered***

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import tensorflow as tf

from tensorflow.keras.models import Model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

import joblib


# ============================================================
# PATHS
# ============================================================

DATASET_PATH = Path(
    "/kaggle/input/datasets/yadavmohit2000/combine-apple"
)

CLEAN_FOLDER = DATASET_PATH / "Clean"
ADV_FOLDER = DATASET_PATH / "Adversarial"

RESNET_PATH = (
    "/kaggle/working/resnet50_6class_20epochs.keras"
)

IMAGE_SIZE = (224, 224)

RANDOM_STATE = 42

K = 5

VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ============================================================
# LOAD TRAINED RESNET50
# ============================================================

print("=" * 70)
print("LOADING RESNET50")
print("=" * 70)

resnet = tf.keras.models.load_model(
    RESNET_PATH
)

print("ResNet50 loaded successfully.")


# ============================================================
# CHECK MODEL
# ============================================================

print("\nResNet input shape:")
print(resnet.input_shape)

print("\nResNet output shape:")
print(resnet.output_shape)


# ============================================================
# FEATURE EXTRACTION LAYER
# ============================================================

feature_layer = None

for layer in reversed(resnet.layers):

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        feature_layer = layer
        break


if feature_layer is None:

    raise ValueError(
        "GlobalAveragePooling2D layer not found."
    )


feature_extractor = Model(
    inputs=resnet.input,
    outputs=feature_layer.output
)


print("\nFeature extraction layer:")
print(feature_layer.name)

print(
    "Feature output shape:",
    feature_extractor.output_shape
)


# ============================================================
# FUNCTION: GET IMAGES
# ============================================================

def get_images(folder):

    folder = Path(folder)

    if not folder.exists():
        return []

    return sorted(
        [
            file
            for file in folder.iterdir()
            if file.is_file()
            and file.suffix.lower()
            in VALID_EXTENSIONS
        ]
    )


# ============================================================
# FUNCTION: EXTRACT RESNET FEATURES
# ============================================================

def extract_features(folder, label):

    images = get_images(folder)

    features = []
    labels = []

    print("\n")
    print("-" * 70)
    print("Folder:", folder)
    print("Images:", len(images))
    print("Label :", label)
    print("-" * 70)

    for i, image_path in enumerate(images):

        try:

            # ------------------------------------------------
            # Load image
            # ------------------------------------------------

            image = Image.open(
                image_path
            ).convert("RGB")


            # ------------------------------------------------
            # Resize
            # ------------------------------------------------

            image = image.resize(
                IMAGE_SIZE
            )


            # ------------------------------------------------
            # Convert to numpy
            # ------------------------------------------------

            image_array = np.asarray(
                image,
                dtype=np.float32
            )


            # ------------------------------------------------
            # Same /255 preprocessing
            # used during ResNet training
            # ------------------------------------------------

            image_array = (
                image_array / 255.0
            )


            # ------------------------------------------------
            # Add batch dimension
            # ------------------------------------------------

            image_array = np.expand_dims(
                image_array,
                axis=0
            )


            # ------------------------------------------------
            # Extract deep feature
            # ------------------------------------------------

            feature = (
                feature_extractor.predict(
                    image_array,
                    verbose=0
                )
            )


            # ------------------------------------------------
            # Flatten feature
            # ------------------------------------------------

            feature = feature.flatten()


            # ------------------------------------------------
            # Store feature and label
            # ------------------------------------------------

            features.append(
                feature
            )

            labels.append(
                label
            )


            if (
                (i + 1) % 50 == 0
                or i == len(images) - 1
            ):

                print(
                    f"Processed "
                    f"{i + 1}/"
                    f"{len(images)}"
                )


        except Exception as e:

            print(
                f"Error processing "
                f"{image_path}: {e}"
            )


    return (
        np.array(features),
        np.array(labels)
    )


# ============================================================
# EXTRACT CLEAN FEATURES
# LABEL = 0
# ============================================================

X_clean, y_clean = extract_features(
    CLEAN_FOLDER,
    label=0
)


# ============================================================
# EXTRACT ADVERSARIAL FEATURES
# LABEL = 1
# ============================================================

X_adv, y_adv = extract_features(
    ADV_FOLDER,
    label=1
)


# ============================================================
# COMBINE FEATURES
# ============================================================

X = np.concatenate(
    [
        X_clean,
        X_adv
    ],
    axis=0
)

y = np.concatenate(
    [
        y_clean,
        y_adv
    ],
    axis=0
)


# ============================================================
# DATASET INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("POST DETECTOR DATASET")
print("=" * 70)

print(
    "Clean images       :",
    len(X_clean)
)

print(
    "Adversarial images :",
    len(X_adv)
)

print(
    "Total images       :",
    len(X)
)

print(
    "Feature dimension  :",
    X.shape[1]
)


# ============================================================
# EXPECTED FEATURE DIMENSION
# ============================================================

print("\n")
print(
    "Expected ResNet feature dimension "
    "if GAP is used: 2048"
)


# ============================================================
# STANDARDIZATION
# ============================================================

print("\n")
print("=" * 70)
print("STANDARDIZING RESNET FEATURES")
print("=" * 70)

scaler_post = StandardScaler()

X_scaled = scaler_post.fit_transform(
    X
)

print(
    "Feature standardization completed."
)

print(
    "Scaled feature shape:",
    X_scaled.shape
)


# ============================================================
# TRAIN POST KNN
# WITH STANDARDIZATION
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING POST KNN")
print("=" * 70)

knn_post = KNeighborsClassifier(
    n_neighbors=K,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)


# ------------------------------------------------------------
# TRAIN KNN ON STANDARDIZED RESNET FEATURES
# ------------------------------------------------------------

knn_post.fit(
    X_scaled,
    y
)

print(
    "POST KNN training completed."
)

print(
    "K =",
    K
)

print(
    "Standardization: USED"
)


# ============================================================
# TRAINING-SET EVALUATION
# ============================================================

y_train_pred = knn_post.predict(
    X_scaled
)

y_train_prob = knn_post.predict_proba(
    X_scaled
)[:, 1]


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_train = confusion_matrix(
    y,
    y_train_pred,
    labels=[0, 1]
)

TN, FP, FN, TP = cm_train.ravel()


# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    y,
    y_train_pred
)

precision = precision_score(
    y,
    y_train_pred,
    zero_division=0
)

recall = recall_score(
    y,
    y_train_pred,
    zero_division=0
)

f1 = f1_score(
    y,
    y_train_pred,
    zero_division=0
)

tpr = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)

fpr = (
    FP / (FP + TN)
    if (FP + TN) > 0
    else 0
)

roc_auc = roc_auc_score(
    y,
    y_train_prob
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("POST KNN TRAINING-SET SANITY CHECK")
print("=" * 70)

print("\nConfusion Matrix:")
print(cm_train)

print("\nTN:", TN)
print("FP:", FP)
print("FN:", FN)
print("TP:", TP)

print("\nAccuracy :", f"{accuracy:.4f}")
print("Precision:", f"{precision:.4f}")
print("Recall   :", f"{recall:.4f}")
print("F1 Score :", f"{f1:.4f}")
print("TPR      :", f"{tpr:.4f}")
print("FPR      :", f"{fpr:.4f}")
print("ROC-AUC  :", f"{roc_auc:.4f}")


# ============================================================
# SAVE POST KNN
# ============================================================

knn_path = (
    "/kaggle/working/"
    "post_knn_resnet_features.pkl"
)

joblib.dump(
    knn_post,
    knn_path
)


# ============================================================
# SAVE STANDARD SCALER
# ============================================================

scaler_path = (
    "/kaggle/working/"
    "post_knn_scaler.pkl"
)

joblib.dump(
    scaler_post,
    scaler_path
)


# ============================================================
# SAVE TRAINING FEATURES
# ============================================================

np.save(
    "/kaggle/working/post_features.npy",
    X
)

np.save(
    "/kaggle/working/post_labels.npy",
    y
)


# ============================================================
# FINAL INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("POST MODEL SAVED")
print("=" * 70)

print(
    "KNN:",
    knn_path
)

print(
    "Scaler:",
    scaler_path
)

print(
    "Features:",
    "/kaggle/working/post_features.npy"
)

print(
    "Labels:",
    "/kaggle/working/post_labels.npy"
)

print("\nFeature files saved.")

In [ ]:

import os
import cv2
import numpy as np

#train_folder = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Tomato/Train'
test_folder = '/kaggle/input/datasets/yadavmohit2000/misxed-5-crop-31/Mixed_dataset/Apple_test'
#test_folder1 = '/kaggle/input/datasets/tushar5harma/plant-village-dataset-updated/Tomato/Test'

IMG_SIZE = (224, 224)
MAX_PER_CLASS = 10000   # 🔴 IMPORTANT

def load_images_limited(folder, max_per_class=None):
    images = []
    labels = []
    
    for label in sorted(os.listdir(folder)):
        label_path = os.path.join(folder, label)
        if not os.path.isdir(label_path):
            continue
        
        count = 0
        for filename in os.listdir(label_path):
            if max_per_class is not None and count >= max_per_class:
                break
            
            img_path = os.path.join(label_path, filename)
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            img = cv2.resize(img, IMG_SIZE)
            images.append(img)
            labels.append(label)
            count += 1

        print(f"Loaded {count} images from class '{label}'")
    
    return np.array(images), np.array(labels)

# 🔹 TRAIN: only 500 images per class
x_train, y_train = load_images_limited(train_folder, MAX_PER_CLASS)

# 🔹 TEST (no limit)
x_test, y_test = load_images_limited(test_folder)
#x_test_1, y_test_1 = load_images_limited(test_folder1)

# 🔹 Verify shapes
print("\nFinal Shapes:")
#print("x_train:", x_train.shape)
#print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)
#print("x_test_1:", x_test_1.shape)
#print("y_test_1:", y_test_1.shape)



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels in y_train
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform the labels in y_test
y_test_encoded = label_encoder.transform(y_test)
#y_test_encoded_1 = label_encoder.transform(y_test_1)





In [ ]:
y_test= y_test_encoded

In [ ]:
import numpy as np
import tensorflow as tf
import joblib

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.preprocessing import StandardScaler


# ============================================================
# STEP 1: ORIGINAL x_test AND y_test
# ============================================================

# x_test = original test images
# y_test = original encoded class labels

# Example:
# x_test.shape = (104, 224, 224, 3)
# y_test.shape = (104,)

# IMPORTANT:
# x_test and y_test already available hone chahiye.
# ============================================================

print("=" * 70)
print("ORIGINAL TEST DATA")
print("=" * 70)

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

if len(x_test) != len(y_test):
    raise ValueError(
        "x_test and y_test must have the same number of samples."
    )


# ============================================================
# STEP 2: LOAD TRAINED RESNET50
# ============================================================

RESNET_PATH = (
    "/kaggle/working/resnet50_6class_20epochs.keras"
)

print("\n" + "=" * 70)
print("LOADING TRAINED RESNET50")
print("=" * 70)

resnet = tf.keras.models.load_model(
    RESNET_PATH
)

print("ResNet50 loaded successfully.")
print("Input shape :", resnet.input_shape)
print("Output shape:", resnet.output_shape)


# ============================================================
# STEP 3: CREATE RESNET FEATURE EXTRACTOR
# ============================================================

# GlobalAveragePooling2D ke output se features niklenge.
# Standard ResNet50 mein normally 2048 features milte hain.
# ============================================================

feature_layer = None

for layer in reversed(resnet.layers):

    if isinstance(
        layer,
        tf.keras.layers.GlobalAveragePooling2D
    ):
        feature_layer = layer
        break


if feature_layer is None:
    raise ValueError(
        "GlobalAveragePooling2D layer not found."
    )


feature_extractor = tf.keras.models.Model(
    inputs=resnet.input,
    outputs=feature_layer.output
)


print("\nFeature extraction layer:")
print(feature_layer.name)

print(
    "Feature extractor output:",
    feature_extractor.output_shape
)


# ============================================================
# STEP 4: PREPROCESS x_test FOR RESNET
# ============================================================

# Original x_test ko change nahi karenge.
# Sirf ResNet ke liye separate variable banega.

# ResNet training mein /255 use hua tha.
# ============================================================

x_test_for_resnet = (
    x_test.astype(np.float32) / 255.0
)

print("\n" + "=" * 70)
print("TEST IMAGE PREPROCESSING")
print("=" * 70)

print(
    "ResNet input shape:",
    x_test_for_resnet.shape
)


# ============================================================
# STEP 5: EXTRACT FEATURES FROM ALL x_test
# ============================================================

print("\n" + "=" * 70)
print("EXTRACTING RESNET FEATURES")
print("=" * 70)

X_test_features = feature_extractor.predict(
    x_test_for_resnet,
    batch_size=32,
    verbose=1
)


# ============================================================
# STEP 6: FLATTEN FEATURES
# ============================================================

X_test_features = X_test_features.reshape(
    X_test_features.shape[0],
    -1
)

print("\nFeature matrix shape:")
print(X_test_features.shape)


# ============================================================
# STEP 7: LOAD TRAINED KNN AND SCALER
# ============================================================

KNN_PATH = (
    "/kaggle/working/"
    "post_knn_resnet_features.pkl"
)

SCALER_PATH = (
    "/kaggle/working/"
    "post_knn_scaler.pkl"
)

print("\n" + "=" * 70)
print("LOADING TRAINED KNN")
print("=" * 70)

knn_post = joblib.load(
    KNN_PATH
)

scaler_post = joblib.load(
    SCALER_PATH
)

print("KNN loaded successfully.")
print("Scaler loaded successfully.")
print("Standardization: USED")


# ============================================================
# STEP 8: STANDARDIZE TEST FEATURES
# ============================================================

# IMPORTANT:
# Yahan fit/fit_transform nahi karna hai.
# Training mein fitted scaler se sirf transform karna hai.
# ============================================================

X_test_features_scaled = scaler_post.transform(
    X_test_features
)

print(
    "\nStandardized test feature shape:",
    X_test_features_scaled.shape
)


# ============================================================
# STEP 9: KNN PREDICTION
# ============================================================

# KNN output:
#
# 0 = CLEAN
# 1 = ADVERSARIAL
#
# IMPORTANT:
# y_test yahan KNN ko nahi diya ja raha.
# ============================================================

print("\n" + "=" * 70)
print("KNN TEST PREDICTION")
print("=" * 70)

knn_prediction = knn_post.predict(
    X_test_features_scaled
)


# ============================================================
# STEP 10: ORIGINAL INDEXING
# ============================================================

all_indices = np.arange(
    len(x_test)
)


# ------------------------------------------------------------
# KNN predicted adversarial indices
# ------------------------------------------------------------

adversarial_indices = all_indices[
    knn_prediction == 1
]


# ------------------------------------------------------------
# KNN predicted clean indices
# ------------------------------------------------------------

clean_indices = all_indices[
    knn_prediction == 0
]


print("\n" + "=" * 70)
print("KNN DETECTION RESULT")
print("=" * 70)

print(
    "Total test images:",
    len(x_test)
)

print(
    "KNN predicted Clean:",
    len(clean_indices)
)

print(
    "KNN predicted Adversarial:",
    len(adversarial_indices)
)

print("\nAdversarial indices:")
print(adversarial_indices)

print("\nClean indices:")
print(clean_indices)


# ============================================================
# STEP 11: DELETE ADVERSARIAL INDICES
# ============================================================

# VERY IMPORTANT:
#
# SAME adversarial indices ko:
#
#   x_test
#   y_test
#
# dono se remove karenge.
#
# Isse image-label correspondence maintain rahegi.
# ============================================================

x_test_clean = np.delete(
    x_test,
    adversarial_indices,
    axis=0
)

y_test_clean = np.delete(
    y_test,
    adversarial_indices,
    axis=0
)


# ============================================================
# STEP 12: VERIFY FILTERED DATA
# ============================================================

print("\n" + "=" * 70)
print("AFTER KNN FILTERING")
print("=" * 70)

print(
    "Original x_test:",
    x_test.shape
)

print(
    "Original y_test:",
    y_test.shape
)

print(
    "Filtered x_test:",
    x_test_clean.shape
)

print(
    "Filtered y_test:",
    y_test_clean.shape
)


# ============================================================
# STEP 13: VERIFY INDEXING
# ============================================================

# Ye check karta hai ki:
#
# Remaining samples = KNN predicted CLEAN samples
# ============================================================

expected_clean_count = len(
    clean_indices
)

actual_clean_count = len(
    x_test_clean
)

if expected_clean_count != actual_clean_count:

    raise ValueError(
        "Filtering error: clean sample count mismatch."
    )

print(
    "\nIndex filtering verified successfully."
)


# ============================================================
# STEP 14: PREPARE ONLY KNN-PREDICTED CLEAN DATA
# ============================================================

# Ab sirf remaining clean images ko ResNet classifier
# ko diya jayega.
#
# Original y_test ke corresponding labels already
# y_test_clean mein preserved hain.
# ============================================================

x_test_clean_for_resnet = (
    x_test_clean.astype(np.float32)
)


# ============================================================
# STEP 15: RESNET CLASSIFICATION
# ============================================================

print("\n" + "=" * 70)
print("TESTING KNN-FILTERED CLEAN DATA ON RESNET50")
print("=" * 70)

resnet_probabilities = resnet.predict(
    x_test_clean_for_resnet,
    batch_size=32,
    verbose=1
)


# ============================================================
# STEP 16: RESNET PREDICTED CLASS
# ============================================================

resnet_predictions = np.argmax(
    resnet_probabilities,
    axis=1
)


# ============================================================
# STEP 17: FINAL CLASSIFICATION ACCURACY
# ============================================================

# Compare:
#
# ResNet prediction
#       VS
# Original corresponding y_test labels
# ============================================================

final_accuracy = accuracy_score(
    y_test_clean,
    resnet_predictions
)


# ============================================================
# STEP 18: FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "Original test images:",
    len(x_test)
)

print(
    "Detected adversarial:",
    len(adversarial_indices)
)

print(
    "Remaining clean:",
    len(x_test_clean)
)

print(
    "Final ResNet Accuracy:",
    f"{final_accuracy:.4f}"
)

print(
    "Final ResNet Accuracy:",
    f"{final_accuracy * 100:.2f}%"
)


# ============================================================
# STEP 19: CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test_clean,
    resnet_predictions
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# STEP 20: CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test_clean,
        resnet_predictions,
        zero_division=0
    )
)


# ============================================================
# STEP 21: SAVE FILTERED DATA
# ============================================================

np.save(
    "/kaggle/working/x_test_clean_after_knn.npy",
    x_test_clean
)

np.save(
    "/kaggle/working/y_test_clean_after_knn.npy",
    y_test_clean
)

np.save(
    "/kaggle/working/adversarial_indices.npy",
    adversarial_indices
)

np.save(
    "/kaggle/working/knn_test_predictions.npy",
    knn_prediction
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PROCESS COMPLETED")
print("=" * 70)

print(
    "Original samples      :",
    len(x_test)
)

print(
    "Adversarial removed   :",
    len(adversarial_indices)
)

print(
    "Clean samples         :",
    len(x_test_clean)
)

print(
    "Final ResNet accuracy :",
    f"{final_accuracy * 100:.2f}%"
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve,
    auc,
    roc_auc_score,
    confusion_matrix
)

class_names = label_encoder.classes_

print("Class names:")
print(class_names)



# ============================================================
# COLOURFUL CONFUSION MATRIX - RESNET50
# ============================================================

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.title(
    "Confusion Matrix - ResNet50"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/resnet_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# MULTICLASS ROC CURVE - RESNET50
# CLASS NAMES FROM y_train
# ============================================================

num_classes = resnet_probabilities.shape[1]

y_test_clean_binary = label_binarize(
    y_test_clean,
    classes=np.arange(num_classes)
)

plt.figure(figsize=(8, 6))

auc_values = {}

for i in range(num_classes):

    fpr, tpr, _ = roc_curve(
        y_test_clean_binary[:, i],
        resnet_probabilities[:, i]
    )

    class_auc = auc(
        fpr,
        tpr
    )

    auc_values[i] = class_auc

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{class_names[i]} (AUC = {class_auc:.4f})"
    )


# Random classifier

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random Classifier"
)


# Macro AUC

macro_auc = roc_auc_score(
    y_test_clean_binary,
    resnet_probabilities,
    multi_class="ovr",
    average="macro"
)

print("\n" + "=" * 70)
print("RESNET ROC-AUC RESULTS")
print("=" * 70)

for i in range(num_classes):

    print(
        f"{class_names[i]} : "
        f"AUC = {auc_values[i]:.4f}"
    )

print(
    f"\nMacro Average AUC : {macro_auc:.4f}"
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "Multiclass ROC Curve - ResNet50"
)

plt.legend(
    loc="lower right",
    fontsize=9
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/resnet_multiclass_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()